In [79]:
from __future__ import annotations

import json
import csv
import math
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Callable, Iterable, Sequence
from uuid import uuid4

from jinja2 import Template

import gmsh
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset


In [80]:
class MeshGenerator:
    """Build a quad-dominant Gmsh mesh with local refinement near fin walls."""

    def __init__(
        self,
        element_size_um: float = 100.0 / 3.0,
        size_min_um: float | None = None,
        size_max_um: float | None = None,
        refine_dist_mm: float | None = None,
        refine_outer_walls: bool = False,
        verbose: bool = False,
    ):
        if element_size_um <= 0:
            raise ValueError("element_size_um must be positive")
        self.pixel_size_m = element_size_um / 1_000_000.0
        self.size_min_m = None if size_min_um is None else size_min_um / 1_000_000.0
        self.size_max_m = None if size_max_um is None else size_max_um / 1_000_000.0
        self.refine_dist_m = None if refine_dist_mm is None else refine_dist_mm / 1000.0
        self.refine_outer_walls = bool(refine_outer_walls)
        self.verbose = bool(verbose)

    def to_dict(self):
        return {
            "element_size_um": self.pixel_size_m * 1_000_000.0,
            "size_min_um": None if self.size_min_m is None else self.size_min_m * 1_000_000.0,
            "size_max_um": None if self.size_max_m is None else self.size_max_m * 1_000_000.0,
            "refine_dist_mm": None if self.refine_dist_m is None else self.refine_dist_m * 1000.0,
            "refine_outer_walls": self.refine_outer_walls,
            "verbose": self.verbose,
        }

    def snap_to_design_pixel(self, value: float):
        if self.pixel_size_m <= 0:
            raise ValueError("pixel size must be positive")
        return round(value / self.pixel_size_m) * self.pixel_size_m

    @staticmethod
    def _classify_boundary_curves(fluid_surfaces, Lx: float, Ly: float, tol: float = 1e-6):
        """Classify unique fluid-boundary curves by their entire bounding box."""
        groups = {"Inlet": [], "Outlet": [], "BottomWall": [], "Symmetry": [], "FinWalls": []}
        seen = set()
        boundary = gmsh.model.getBoundary(
            [(2, s) for s in fluid_surfaces], combined=True, oriented=False
        )
        for dim, tag in boundary:
            if tag in seen:
                continue
            seen.add(tag)
            xmin, ymin, _, xmax, ymax, _ = gmsh.model.getBoundingBox(dim, tag)
            if abs(xmin) <= tol and abs(xmax) <= tol:
                groups["Inlet"].append(tag)
            elif abs(xmin - Lx) <= tol and abs(xmax - Lx) <= tol:
                groups["Outlet"].append(tag)
            elif abs(ymin) <= tol and abs(ymax) <= tol:
                groups["BottomWall"].append(tag)
            elif abs(ymin - Ly) <= tol and abs(ymax - Ly) <= tol:
                groups["Symmetry"].append(tag)
            else:
                groups["FinWalls"].append(tag)
        return groups

    def generate_mesh(
        self,
        Lx: float,
        Ly: float,
        nx: int,
        ny: int,
        fin_coords: Iterable[Sequence[Sequence[float]]],
        design_id: str,
        output_path: str,
        *,
        size_min: float | None = None,
        size_max: float | None = None,
        refine_dist: float | None = None,
        refine_outer_walls: bool | None = None,
        pixel_m: float | None = None,
        verbose: bool | None = None,
    ) -> str:
        """Create a 2D quad-dominant mesh, refined near fin and selected walls.

        ``size_min`` and ``size_max`` describe target element sizes directly.
        The structured design grid and the CFD mesh are intentionally distinct.
        Fins are polygons removed from the fluid rectangle; every polygon vertex
        must lie on the design pixel grid.
        """
        pixel_m = self.pixel_size_m if pixel_m is None else pixel_m
        size_min = self.size_min_m if size_min is None else size_min
        size_max = self.size_max_m if size_max is None else size_max
        refine_dist = self.refine_dist_m if refine_dist is None else refine_dist
        refine_outer_walls = self.refine_outer_walls if refine_outer_walls is None else refine_outer_walls
        verbose = self.verbose if verbose is None else verbose
        if Lx <= 0 or Ly <= 0 or nx < 1 or ny < 1:
            raise ValueError("Lx/Ly must be positive and nx/ny must be >= 1")
        dx, dy = Lx / nx, Ly / ny
        if size_min is None:
            size_min = min(dx, dy) / 2
        if size_max is None:
            size_max = max(dx, dy)
        if refine_dist is None:
            refine_dist = max(Lx, Ly) / 10
        if not (0 < size_min <= size_max) or refine_dist <= 0:
            raise ValueError("Require 0 < size_min <= size_max and refine_dist > 0")

        fin_coords = list(fin_coords)
        # Enforce pixel alignment to avoid silently changing the printable layout.
        for poly in fin_coords:
            for point in poly:
                for coordinate in point[:2]:
                    if abs(coordinate - round(coordinate / pixel_m) * pixel_m) > 1e-9:
                        raise ValueError(f"Fin coordinate {coordinate} is off the design pixel grid")

        gmsh.initialize()
        try:
            gmsh.option.setNumber("General.Terminal", 1 if verbose else 0)
            gmsh.model.add(f"cold_plate_{design_id}")

            rect = gmsh.model.occ.addRectangle(0, 0, 0, Lx, Ly)
            obstructions = []
            for polygon in fin_coords:
                points = [gmsh.model.occ.addPoint(x, y, 0) for x, y in polygon]
                lines = [gmsh.model.occ.addLine(points[i], points[(i + 1) % len(points)])
                         for i in range(len(points))]
                loop = gmsh.model.occ.addCurveLoop(lines)
                obstructions.append(gmsh.model.occ.addPlaneSurface([loop]))
            gmsh.model.occ.synchronize()

            if obstructions:
                cut, _ = gmsh.model.occ.cut([(2, rect)], [(2, s) for s in obstructions],
                                            removeObject=True, removeTool=False)
                gmsh.model.occ.synchronize()
                fluid_surfaces = [tag for dim, tag in cut if dim == 2]
                # Keep the original fin surfaces so the output mesh includes both
                # material regions for the Test 1 temperature solve.
                existing_surface_tags = {tag for dim, tag in gmsh.model.getEntities(2)}
                solid_surfaces = [tag for tag in obstructions
                                  if tag in existing_surface_tags]
            else:
                fluid_surfaces = [rect]
                solid_surfaces = []
            if not fluid_surfaces:
                raise RuntimeError("Boolean subtraction produced no fluid surfaces")

            fluid_group = gmsh.model.addPhysicalGroup(2, fluid_surfaces)
            gmsh.model.setPhysicalName(2, fluid_group, "Fluid")
            if solid_surfaces:
                solid_group = gmsh.model.addPhysicalGroup(2, solid_surfaces)
                gmsh.model.setPhysicalName(2, solid_group, "Solid")
            groups = self._classify_boundary_curves(fluid_surfaces, Lx, Ly)
            for name, tags in groups.items():
                if tags:
                    group = gmsh.model.addPhysicalGroup(1, tags)
                    gmsh.model.setPhysicalName(1, group, name)

            # Size is driven only by the background field to avoid point/curvature
            # sizing overriding wall refinement. Use these sizes directly; Gmsh's
            # subdivision algorithm can fail when a 1D edge has an odd segment count.
            h_min, h_max = size_min, size_max
            gmsh.option.setNumber("Mesh.MeshSizeFromPoints", 0)
            gmsh.option.setNumber("Mesh.MeshSizeFromCurvature", 0)
            gmsh.option.setNumber("Mesh.MeshSizeExtendFromBoundary", 0)
            gmsh.option.setNumber("Mesh.MeshSizeMin", h_min)
            gmsh.option.setNumber("Mesh.MeshSizeMax", h_max)

            refine_curves = list(groups["FinWalls"])
            if refine_outer_walls:
                refine_curves.extend(groups["BottomWall"])
                refine_curves.extend(groups["Symmetry"])
            if refine_curves:
                distance = gmsh.model.mesh.field.add("Distance")
                gmsh.model.mesh.field.setNumbers(distance, "CurvesList", refine_curves)
                gmsh.model.mesh.field.setNumber(distance, "Sampling", 100)
                threshold = gmsh.model.mesh.field.add("Threshold")
                gmsh.model.mesh.field.setNumber(threshold, "InField", distance)
                gmsh.model.mesh.field.setNumber(threshold, "SizeMin", h_min)
                gmsh.model.mesh.field.setNumber(threshold, "SizeMax", h_max)
                gmsh.model.mesh.field.setNumber(threshold, "DistMin", 0)
                gmsh.model.mesh.field.setNumber(threshold, "DistMax", refine_dist)
                gmsh.model.mesh.field.setAsBackgroundMesh(threshold)

            # Start with Frontal-Delaunay and Blossom recombination. Blossom can
            # leave triangles where creating a quad would produce poor elements,
            # so subdivide the resulting hybrid mesh into all quadrilaterals.
            # This is required because Exodus/MOOSE cannot write QUAD4 and TRI3
            # together in the same material block.
            gmsh.option.setNumber("Mesh.Algorithm", 6)
            gmsh.option.setNumber("Mesh.RecombinationAlgorithm", 1)
            gmsh.option.setNumber("Mesh.RecombineAll", 1)
            gmsh.option.setNumber("Mesh.ElementOrder", 1)
            gmsh.model.mesh.generate(2)

            # Gmsh's subdivision algorithm 1 converts every 2D element to
            # quadrilaterals. Apply it explicitly to the generated mesh.
            gmsh.option.setNumber("Mesh.SubdivisionAlgorithm", 1)
            gmsh.model.mesh.refine()

            element_types, element_tags, _ = gmsh.model.mesh.getElements(2)
            if not any(len(tags) for tags in element_tags):
                raise RuntimeError("Gmsh generated no 2D elements")
            element_counts = {
                gmsh.model.mesh.getElementProperties(element_type)[0]: len(tags)
                for element_type, tags in zip(element_types, element_tags)
            }
            non_quadrilaterals = [
                name for name, count in element_counts.items()
                if count and not name.lower().startswith("quadrilateral")
            ]
            if non_quadrilaterals:
                raise RuntimeError(
                    "MOOSE requires an all-quad 2D mesh, but Gmsh produced "
                    f"non-quadrilateral element types: {non_quadrilaterals}"
                )
            if verbose:
                print(f"Generated all-quad 2D element counts: {element_counts}")
            gmsh.write(output_path)
            return output_path
        finally:
            gmsh.finalize()


In [81]:
class GeometryUtilities:
    """Internal grid and geometry helpers used by the two public classes."""

    @staticmethod
    def _jsonable(value):
        if isinstance(value, dict):
            return {str(k): GeometryUtilities._jsonable(v) for k, v in value.items()}
        if isinstance(value, (list, tuple)):
            return [GeometryUtilities._jsonable(v) for v in value]
        if isinstance(value, np.generic):
            return value.item()
        if isinstance(value, Path):
            return str(value)
        return value

    @staticmethod
    def _run_components(mask: np.ndarray):
        """Label 4-connected components using row runs; return component summaries.

        Each summary contains the component bounding box and whether it touches
        any domain edge, the inlet edge, and the outlet edge. Run-length labeling
        avoids a Python queue visit for every cell on the high-resolution grid.
        """
        ny, nx = mask.shape
        parent: list[int] = []
        boxes: list[list[int]] = []  # xmin, ymin, xmax, ymax
        run_nodes: list[tuple[int, int, int, int]] = []  # node, x0, x1-exclusive, y

        def find(a: int) -> int:
            while parent[a] != a:
                parent[a] = parent[parent[a]]
                a = parent[a]
            return a

        def union(a: int, b: int) -> None:
            ra, rb = find(a), find(b)
            if ra == rb:
                return
            parent[rb] = ra
            ba, bb = boxes[ra], boxes[rb]
            ba[0] = min(ba[0], bb[0]); ba[1] = min(ba[1], bb[1])
            ba[2] = max(ba[2], bb[2]); ba[3] = max(ba[3], bb[3])

        previous: list[tuple[int, int, int]] = []  # x0, x1-exclusive, node
        for y in range(ny):
            row = mask[y]
            changes = np.diff(np.concatenate(([False], row, [False])).astype(np.int8))
            starts = np.flatnonzero(changes == 1)
            stops = np.flatnonzero(changes == -1)
            current: list[tuple[int, int, int]] = []
            p = 0
            for x0, x1 in zip(starts.tolist(), stops.tolist()):
                node = len(parent)
                parent.append(node)
                boxes.append([x0, y, x1 - 1, y])
                current.append((x0, x1, node))
                while p < len(previous) and previous[p][1] <= x0:
                    p += 1
                q = p
                while q < len(previous) and previous[q][0] < x1:
                    if previous[q][1] > x0:
                        union(node, previous[q][2])
                    q += 1
                run_nodes.append((node, x0, x1, y))
            previous = current

        summaries: dict[int, dict[str, Any]] = {}
        for node, x0, x1, y in run_nodes:
            root = find(node)
            entry = summaries.setdefault(root, {
                "bbox": boxes[root].copy(), "touches_boundary": False,
                "touches_inlet": False, "touches_outlet": False,
            })
            entry["bbox"] = boxes[root].copy()
            entry["touches_inlet"] |= x0 == 0
            entry["touches_outlet"] |= x1 == nx
            entry["touches_boundary"] |= y == 0 or y == ny - 1 or x0 == 0 or x1 == nx
        return list(summaries.values())

    @staticmethod
    def _line_runs(line: np.ndarray):
        changes = np.diff(np.concatenate(([False], line, [False])).astype(np.int8))
        starts = np.flatnonzero(changes == 1)
        stops = np.flatnonzero(changes == -1)
        return zip(starts.tolist(), stops.tolist())

    @staticmethod
    def _mask_to_polygons(solid: np.ndarray, dx_mm: float, dy_mm: float):
        """Convert a pixel solid mask into disjoint boundary polygons in mm."""
        ny, nx = solid.shape
        edges: set[tuple[tuple[int, int], tuple[int, int]]] = set()
        ys, xs = np.nonzero(solid)
        for y, x in zip(ys.tolist(), xs.tolist()):
            if y == 0 or not solid[y - 1, x]:
                edges.add(((x, y), (x + 1, y)))
            if x == nx - 1 or not solid[y, x + 1]:
                edges.add(((x + 1, y), (x + 1, y + 1)))
            if y == ny - 1 or not solid[y + 1, x]:
                edges.add(((x + 1, y + 1), (x, y + 1)))
            if x == 0 or not solid[y, x - 1]:
                edges.add(((x, y + 1), (x, y)))

        outgoing: dict[tuple[int, int], list[tuple[int, int]]] = defaultdict(list)
        for a, b in edges:
            outgoing[a].append(b)
        direction = {(1, 0): 0, (0, 1): 1, (-1, 0): 2, (0, -1): 3}
        polygons = []
        unused = set(edges)
        while unused:
            start_edge = next(iter(unused))
            start, current = start_edge
            loop = [start]
            previous = start
            unused.remove(start_edge)
            guard = 0
            while current != start:
                loop.append(current)
                incoming = direction[(current[0] - previous[0], current[1] - previous[1])]
                candidates = [end for end in outgoing[current] if (current, end) in unused]
                if not candidates:
                    raise ValueError("Could not close a solid boundary loop")
                # At a grid vertex, prefer the leftmost turn to keep diagonal
                # contacts as separate loops rather than creating self-crossings.
                def turn_rank(end):
                    d = direction[(end[0] - current[0], end[1] - current[1])]
                    delta = (d - incoming) % 4
                    return {1: 0, 0: 1, 3: 2, 2: 3}[delta]
                nxt = min(candidates, key=turn_rank)
                unused.remove((current, nxt))
                previous, current = current, nxt
                guard += 1
                if guard > len(edges) + 1:
                    raise ValueError("Boundary tracing exceeded its safety limit")

            # Drop collinear grid vertices before creating OCC curves.
            compact = []
            for i, point in enumerate(loop):
                prev, nxt = loop[i - 1], loop[(i + 1) % len(loop)]
                if (point[0] - prev[0]) * (nxt[1] - point[1]) == \
                   (point[1] - prev[1]) * (nxt[0] - point[0]):
                    continue
                compact.append(point)
            if len(compact) >= 3:
                polygons.append([(x * dx_mm, y * dy_mm) for x, y in compact])
        return polygons



In [82]:
@dataclass
class HeatExchanger:
    """One design: geometry, metadata, generated mesh path, and result fields."""

    occupancy_grid: np.ndarray
    fin_plate_x_mm: float = 54.1
    fin_plate_y_mm: float = 52.15
    element_size_mm: float = (100.0 / 3.0) / 1000.0
    design_id: str | None = None
    base_dir: str | Path = "."
    results: dict[str, Any] | None = None
    mesh_path: str | None = None
    mesh_generator: MeshGenerator | None = None
    metadata: dict[str, Any] | None = None

    def __post_init__(self):
        self.occupancy_grid = np.asarray(self.occupancy_grid, dtype=bool)
        if self.occupancy_grid.ndim != 2:
            raise ValueError("occupancy_grid must be a 2D array")
        if self.element_size_mm <= 0:
            raise ValueError("element_size_mm must be positive")
        if self.design_id is None:
            self.design_id = uuid4().hex[:12]
        self.base_dir = Path(self.base_dir)
        self.results = dict(self.results or {})
        self.metadata = dict(self.metadata or {})
        if self.mesh_generator is None:
            self.mesh_generator = MeshGenerator(element_size_um=self.element_size_mm * 1000.0)

    @property
    def grid_ny(self) -> int:
        return int(self.occupancy_grid.shape[0])

    @property
    def grid_nx(self) -> int:
        return int(self.occupancy_grid.shape[1])

    @property
    def actual_dimensions_mm(self) -> tuple[float, float]:
        return self.grid_nx * self.element_size_mm, self.grid_ny * self.element_size_mm

    def generate_mesh(self, output_path: str | Path | None = None, **mesh_options) -> str:
        """Generate a MOOSE-compatible Gmsh mesh from the stored occupancy grid."""
        if output_path is None:
            mesh_dir = self.base_dir / "meshes"
            mesh_dir.mkdir(parents=True, exist_ok=True)
            output_path = mesh_dir / f"{self.design_id}.msh"
        else:
            output_path = Path(output_path)
            output_path.parent.mkdir(parents=True, exist_ok=True)
        mesh_options.setdefault("pixel_m", self.element_size_mm / 1000.0)
        dx = dy = self.element_size_mm
        fin_polygons = GeometryUtilities._mask_to_polygons(self.occupancy_grid, dx, dy)
        Lx, Ly = self.actual_dimensions_mm
        result = self.mesh_generator.generate_mesh(
            Lx=Lx / 1000.0, Ly=Ly / 1000.0,
            nx=self.grid_nx, ny=self.grid_ny,
            fin_coords=[[(x / 1000.0, y / 1000.0) for x, y in p] for p in fin_polygons],
            design_id=self.design_id, output_path=str(output_path),
            **mesh_options,
        )
        self.mesh_path = str(result)
        return self.mesh_path

    def read_moose_results(self, result_path: str | Path | None = None):
        """Stub: Exodus result reading will be implemented with the MOOSE setup."""
        pass

    def save(self, path: str | Path | None = None) -> str:
        """Save geometry, metadata, mesh reference, and results in one NPZ file."""
        if path is None:
            path = self.base_dir / f"{self.design_id}.npz"
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        meta = {
            "design_id": self.design_id,
            "fin_plate_x_mm": float(self.fin_plate_x_mm),
            "fin_plate_y_mm": float(self.fin_plate_y_mm),
            "element_size_mm": float(self.element_size_mm),
            "mesh_path": self.mesh_path,
            "mesh_config": self.mesh_generator.to_dict(),
            "metadata": GeometryUtilities._jsonable(self.metadata),
            "result_items": [],
        }
        arrays: dict[str, np.ndarray] = {"occupancy_grid": self.occupancy_grid.astype(np.uint8)}
        for i, (key, value) in enumerate(self.results.items()):
            if isinstance(value, (np.ndarray, list, tuple)):
                array_key = f"result_{i}"
                arrays[array_key] = np.asarray(value)
                meta["result_items"].append({"name": key, "array": array_key})
            else:
                try:
                    value_json = GeometryUtilities._jsonable(value)
                    json.dumps(value_json)
                except TypeError as exc:
                    raise TypeError(f"Result {key!r} is not serializable") from exc
                meta["result_items"].append({"name": key, "value": value_json})
        arrays["metadata_json"] = np.asarray(json.dumps(meta))
        with path.open("wb") as stream:
            np.savez_compressed(stream, **arrays)
        return str(path)

    @classmethod
    def read(cls, path: str | Path) -> "HeatExchanger":
        """Reload one saved design without requiring pickle-enabled archives."""
        path = Path(path)
        with np.load(path, allow_pickle=False) as saved:
            meta = json.loads(str(saved["metadata_json"].item()))
            results = {}
            for item in meta.get("result_items", []):
                if "array" in item:
                    results[item["name"]] = saved[item["array"]].copy()
                else:
                    results[item["name"]] = item.get("value")
            return cls(
                occupancy_grid=saved["occupancy_grid"].astype(bool),
                fin_plate_x_mm=meta["fin_plate_x_mm"],
                fin_plate_y_mm=meta["fin_plate_y_mm"],
                element_size_mm=meta["element_size_mm"],
                design_id=meta["design_id"],
                base_dir=path.parent,
                results=results,
                mesh_path=meta.get("mesh_path"),
                mesh_generator=MeshGenerator(**meta.get("mesh_config", {})),
                metadata=meta.get("metadata", {}),
            )

In [83]:
class HeatExchangerCNN(nn.Module):
    """Convolutional VAE that learns a prior over valid occupancy grids.

    Training examples are valid designs from Generator's procedural random
    sampler. The network reconstructs one output value per design-grid cell;
    the selected physical element size is set by Generator, not by CNN pooling.
    """

    def __init__(
        self,
        grid_nx: int,
        grid_ny: int,
        latent_dim: int = 32,
        base_channels: int = 8,
    ):
        super().__init__()
        if min(grid_nx, grid_ny, latent_dim, base_channels) < 1:
            raise ValueError("CNN dimensions and channel counts must be positive")
        self.grid_nx = int(grid_nx)
        self.grid_ny = int(grid_ny)
        self.latent_dim = int(latent_dim)
        self.base_channels = int(base_channels)
        self.grid_h = self.grid_ny
        self.grid_w = self.grid_nx

        # Encode the full-resolution design-grid occupancy image down to a small feature map.
        self.encoder_blocks = nn.ModuleList()
        channels = []
        in_channels = 1
        h, w = self.grid_h, self.grid_w
        while min(h, w) > 8 and len(channels) < 8:
            out_channels = min(base_channels * (2 ** len(channels)), 64)
            self.encoder_blocks.append(nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=4, stride=2, padding=1),
                nn.GroupNorm(math.gcd(8, out_channels), out_channels),
                nn.SiLU(),
            ))
            channels.append(out_channels)
            in_channels = out_channels
            h //= 2
            w //= 2
        if not channels:
            channels = [base_channels]
            in_channels = 1
            self.encoder_blocks.append(nn.Sequential(
                nn.Conv2d(1, base_channels, kernel_size=3, stride=1, padding=1),
                nn.GroupNorm(math.gcd(8, base_channels), base_channels),
                nn.SiLU(),
            ))
            h, w = self.grid_h, self.grid_w

        self.bottleneck_channels = channels[-1]
        self.bottleneck_h = h
        self.bottleneck_w = w
        flattened = self.bottleneck_channels * h * w
        self.to_mu = nn.Linear(flattened, latent_dim)
        self.to_logvar = nn.Linear(flattened, latent_dim)
        self.from_latent = nn.Linear(latent_dim, flattened)

        self.decoder_blocks = nn.ModuleList()
        if len(channels) > 1:
            in_ch = channels[-1]
            for out_ch in reversed(channels):
                self.decoder_blocks.append(nn.Sequential(
                    nn.ConvTranspose2d(in_ch, out_ch, kernel_size=4, stride=2, padding=1),
                    nn.GroupNorm(math.gcd(8, out_ch), out_ch),
                    nn.SiLU(),
                ))
                in_ch = out_ch
            decoder_channels = channels[0]
        else:
            decoder_channels = self.bottleneck_channels
        self.output_head = nn.Conv2d(decoder_channels, 1, kernel_size=3, padding=1)
        self._initialize_weights()

    def _initialize_weights(self):
        for layer in self.modules():
            if isinstance(layer, (nn.Conv2d, nn.ConvTranspose2d, nn.Linear)):
                nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
                if layer.bias is not None:
                    nn.init.zeros_(layer.bias)

    def encode(self, occupancy: torch.Tensor):
        x = occupancy
        for block in self.encoder_blocks:
            x = block(x)
        x = x.flatten(start_dim=1)
        return self.to_mu(x), self.to_logvar(x).clamp(-12.0, 12.0)

    @staticmethod
    def reparameterize(mu: torch.Tensor, logvar: torch.Tensor):
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std

    def decode_logits(self, z: torch.Tensor):
        x = self.from_latent(z).view(
            z.shape[0], self.bottleneck_channels, self.bottleneck_h, self.bottleneck_w
        )
        for block in self.decoder_blocks:
            x = block(x)
        x = self.output_head(x)
        return F.interpolate(x, size=(self.grid_h, self.grid_w), mode="bilinear", align_corners=False)

    def forward(self, occupancy: torch.Tensor):
        mu, logvar = self.encode(occupancy)
        z = self.reparameterize(mu, logvar)
        return self.decode_logits(z), mu, logvar

    def generate_grid(self, grid_ny: int, grid_nx: int, latent):
        """Return full-resolution solid probabilities for Generator."""
        device = next(self.parameters()).device
        z = torch.as_tensor(latent, dtype=torch.float32, device=device).reshape(1, -1)
        was_training = self.training
        self.eval()
        with torch.no_grad():
            probabilities = torch.sigmoid(self.decode_logits(z))
            probabilities = F.interpolate(
                probabilities, size=(grid_ny, grid_nx), mode="bilinear", align_corners=False
            )
        if was_training:
            self.train()
        return probabilities[0, 0].cpu().numpy()


In [84]:
class Generator:
    """Generate and validate grid-aligned cold-plate designs.

    ``neural_model`` is optional. For ``method='neural'`` it must accept a
    latent tensor of shape (1, latent_dim) and return occupancy probabilities
    shaped (ny, nx), (1, ny, nx), or (1, 1, ny, nx). The programmatic method
    creates either an empty channel or a regular array of long horizontal fins.
    """

    def __init__(
        self,
        fin_plate_x_mm: float = 54.1,
        fin_plate_y_mm: float = 52.15,
        element_size_um: float = 100.0 / 3.0,
        *,
        neural_model: Callable | Any | None = None,
        latent_dim: int = 32,
        seed: int | None = None,
        base_dir: str | Path = ".",
        max_attempts: int = 100,
        min_feature_mm: float = 0.100,
        max_fin_thickness_mm: float = 0.500,
        target_solid_fraction: float = 0.20,
        solid_fraction_tolerance: float = 0.05,
        chip_gap_mm: float | None = None,
        mesh_generator: MeshGenerator | None = None,
        model_base_channels: int = 8,
        device: torch.device | str | None = None,
        neural_model_trained: bool = False,
        programmatic_layout: str = "empty",
        programmatic_n_fins: int = 5,
        programmatic_fin_thickness_mm: float = 0.100,
        programmatic_inlet_length_mm: float = 5.0,
        programmatic_outlet_length_mm: float = 5.0,
        fallback_column_count_min: int = 12,
        fallback_column_count_max: int = 24,
        fallback_column_width_variation: float = 0.70,
        fallback_full_height_gash_probability: float = 0.25,
        fallback_partial_gash_min_fraction: float = 0.05,
        fallback_partial_gash_max_fraction: float = 0.10,
    ):
        if fin_plate_x_mm <= 0 or fin_plate_y_mm <= 0 or element_size_um <= 0:
            raise ValueError("Plate dimensions and element size must be positive")
        if latent_dim < 1 or max_attempts < 1:
            raise ValueError("latent_dim and max_attempts must be positive")
        self.min_feature_mm = float(min_feature_mm)
        self.max_fin_thickness_mm = float(max_fin_thickness_mm)
        self.target_solid_fraction = float(target_solid_fraction)
        self.solid_fraction_tolerance = float(solid_fraction_tolerance)
        self.solid_fraction_min = self.target_solid_fraction - self.solid_fraction_tolerance
        self.solid_fraction_max = self.target_solid_fraction + self.solid_fraction_tolerance
        if not 0.0 <= self.target_solid_fraction <= 1.0:
            raise ValueError("target_solid_fraction must be between 0 and 1")
        if self.solid_fraction_tolerance < 0 or self.solid_fraction_min < 0 or self.solid_fraction_max > 1:
            raise ValueError("solid_fraction_tolerance must keep the target range within [0, 1]")
        self.chip_gap_mm = None if chip_gap_mm is None else float(chip_gap_mm)
        if self.min_feature_mm <= 0 or self.max_fin_thickness_mm < self.min_feature_mm:
            raise ValueError("Feature limits must satisfy 0 < minimum <= maximum")
        if self.chip_gap_mm is not None and self.chip_gap_mm < 0:
            raise ValueError("chip_gap_mm must be nonnegative or None")
        self.fin_plate_x_mm = float(fin_plate_x_mm)
        self.fin_plate_y_mm = float(fin_plate_y_mm)
        self.element_size_mm = float(element_size_um) / 1000.0
        self.element_size_um = float(element_size_um)
        if math.ceil(self.min_feature_mm / self.element_size_mm - 1e-10) > \
           math.floor(self.max_fin_thickness_mm / self.element_size_mm + 1e-10):
            raise ValueError("Element size is incompatible with the 0.100-0.500 mm feature limits")
        # Keep the requested plate envelope as an upper bound. If a dimension
        # is not an integer number of pixels, the final row/column is omitted.
        self.grid_nx = int(math.floor(self.fin_plate_x_mm / self.element_size_mm + 1e-10))
        self.grid_ny = int(math.floor(self.fin_plate_y_mm / self.element_size_mm + 1e-10))
        if self.grid_nx < 3 or self.grid_ny < 3:
            raise ValueError("Plate dimensions must contain at least a 3 x 3 pixel grid")
        self.latent_dim = int(latent_dim)
        self.model_base_channels = int(model_base_channels)
        if device is None:
            if torch.cuda.is_available():
                device = torch.device("cuda")
            elif torch.backends.mps.is_available():
                device = torch.device("mps")
            else:
                device = torch.device("cpu")
        self.device = torch.device(device)
        self.neural_model = neural_model
        self.model_trained = bool(neural_model_trained)
        self.stage1_history: list[dict[str, float]] = []
        self.rng = np.random.default_rng(seed)
        self.base_dir = Path(base_dir)
        self.mesh_generator = mesh_generator or MeshGenerator(element_size_um=element_size_um)
        self.max_attempts = int(max_attempts)
        self.last_validation_report: dict[str, Any] | None = None
        self.programmatic_layout = str(programmatic_layout).lower()
        if self.programmatic_layout not in {"empty", "parallel_fins"}:
            raise ValueError("programmatic_layout must be 'empty' or 'parallel_fins'")
        if programmatic_n_fins < 1:
            raise ValueError("programmatic_n_fins must be positive")
        self.programmatic_n_fins = int(programmatic_n_fins)
        self.programmatic_fin_thickness_mm = float(programmatic_fin_thickness_mm)
        self.programmatic_inlet_length_mm = float(programmatic_inlet_length_mm)
        self.programmatic_outlet_length_mm = float(programmatic_outlet_length_mm)
        if not self.min_feature_mm <= self.programmatic_fin_thickness_mm <= self.max_fin_thickness_mm:
            raise ValueError("Programmatic fin thickness must satisfy the configured feature limits")
        if self.programmatic_inlet_length_mm < 0 or self.programmatic_outlet_length_mm < 0:
            raise ValueError("Programmatic inlet/outlet lengths must be nonnegative")
        if fallback_column_count_min < 1 or fallback_column_count_max < fallback_column_count_min:
            raise ValueError("Fallback column counts must satisfy 1 <= minimum <= maximum")
        self.fallback_column_count_min = int(fallback_column_count_min)
        self.fallback_column_count_max = int(fallback_column_count_max)
        self.fallback_column_width_variation = float(fallback_column_width_variation)
        self.fallback_full_height_gash_probability = float(fallback_full_height_gash_probability)
        self.fallback_partial_gash_min_fraction = float(fallback_partial_gash_min_fraction)
        self.fallback_partial_gash_max_fraction = float(fallback_partial_gash_max_fraction)
        if not 0.0 <= self.fallback_column_width_variation <= 1.0:
            raise ValueError("fallback_column_width_variation must be in [0, 1]")
        if not 0.0 <= self.fallback_full_height_gash_probability <= 1.0:
            raise ValueError("fallback_full_height_gash_probability must be in [0, 1]")
        if not (0.0 < self.fallback_partial_gash_min_fraction <=
                self.fallback_partial_gash_max_fraction < 1.0):
            raise ValueError("Partial-gash fractions must satisfy 0 < min <= max < 1")
        self.initialize_model()

    @property
    def actual_dimensions_mm(self) -> tuple[float, float]:
        return self.grid_nx * self.element_size_mm, self.grid_ny * self.element_size_mm

    def initialize_model(self):
        """Create the CNN prior, or move a supplied model onto the selected device."""
        if self.neural_model is None:
            self.neural_model = HeatExchangerCNN(
                grid_nx=self.grid_nx,
                grid_ny=self.grid_ny,
                latent_dim=self.latent_dim,
                base_channels=self.model_base_channels,
            )
            self.model_trained = False
        if hasattr(self.neural_model, "to"):
            self.neural_model.to(self.device)
        return self.neural_model

    def train_stage1(
        self,
        n_samples: int = 64,
        num_epochs: int = 20,
        batch_size: int = 1,
        learning_rate: float = 1e-3,
        beta: float = 1e-3,
        validation_fraction: float = 0.1,
        log_every: int = 1,
    ) -> list[dict[str, float]]:
        """Pretrain the CNN to reproduce valid designs from the random sampler.

        This first pass is a convolutional VAE objective (reconstruction plus
        KL regularization), not the performance-optimizing RL stage. Every
        training example is generated and checked by this Generator first.
        """
        if not isinstance(self.neural_model, HeatExchangerCNN):
            raise TypeError("Stage-one training currently requires HeatExchangerCNN")
        if n_samples < 2 or num_epochs < 1 or batch_size < 1 or learning_rate <= 0 or beta < 0:
            raise ValueError("Invalid stage-one training settings")
        if not 0 <= validation_fraction < 0.5:
            raise ValueError("validation_fraction must be in [0, 0.5)")

        self.initialize_model()
        examples = []
        for sample_index in range(n_samples):
            for _ in range(self.max_attempts):
                grid = self._random_grid()
                valid, report = self.validate(grid)
                if valid:
                    self.last_validation_report = report
                    full = torch.from_numpy(grid.astype(np.float32))[None, None]
                    examples.append(full[0])
                    break
            else:
                raise RuntimeError(f"Could not create valid stage-one sample {sample_index}")

        data = torch.stack(examples)
        order = torch.randperm(len(data))
        n_val = int(round(n_samples * validation_fraction))
        if validation_fraction > 0:
            n_val = min(max(n_val, 1), n_samples - 1)
        else:
            n_val = 0
        val_data = data[order[:n_val]] if n_val else None
        train_data = data[order[n_val:]]
        train_loader = DataLoader(TensorDataset(train_data), batch_size=batch_size, shuffle=True)
        val_loader = (DataLoader(TensorDataset(val_data), batch_size=batch_size, shuffle=False)
                      if val_data is not None else None)

        solid_fraction = float(data.mean().item())
        pos_weight = torch.tensor(
            [(1.0 - solid_fraction) / max(solid_fraction, 1e-3)],
            dtype=torch.float32,
            device=self.device,
        ).clamp(max=30.0)
        optimizer = torch.optim.Adam(self.neural_model.parameters(), lr=learning_rate)

        def loss_for_batch(target):
            target = target.to(self.device)
            logits, mu, logvar = self.neural_model(target)
            reconstruction = F.binary_cross_entropy_with_logits(
                logits, target, pos_weight=pos_weight
            )
            kl = -0.5 * torch.mean(1.0 + logvar - mu.square() - logvar.exp())
            return reconstruction + beta * kl, reconstruction, kl

        history = []
        for epoch in range(1, num_epochs + 1):
            self.neural_model.train()
            totals = torch.zeros(3, dtype=torch.float64)
            batches = 0
            for (target,) in train_loader:
                optimizer.zero_grad(set_to_none=True)
                loss, reconstruction, kl = loss_for_batch(target)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.neural_model.parameters(), max_norm=5.0)
                optimizer.step()
                totals += torch.tensor([loss.item(), reconstruction.item(), kl.item()], dtype=torch.float64)
                batches += 1

            record = {
                "epoch": float(epoch),
                "train_loss": float(totals[0] / max(batches, 1)),
                "train_reconstruction": float(totals[1] / max(batches, 1)),
                "train_kl": float(totals[2] / max(batches, 1)),
            }
            if val_loader is not None:
                self.neural_model.eval()
                val_totals = torch.zeros(3, dtype=torch.float64)
                val_batches = 0
                with torch.no_grad():
                    for (target,) in val_loader:
                        values = loss_for_batch(target)
                        val_totals += torch.tensor([v.item() for v in values], dtype=torch.float64)
                        val_batches += 1
                record["validation_loss"] = float(val_totals[0] / max(val_batches, 1))
            history.append(record)
            if log_every and (epoch % log_every == 0 or epoch == num_epochs):
                print(f"Stage 1 epoch {epoch}/{num_epochs}: loss={record['train_loss']:.4f}")

        self.neural_model.eval()
        self.model_trained = True
        self.stage1_history = history
        return history

    def validate(
        self,
        occupancy_grid: np.ndarray,
        *,
        enforce_solid_fraction: bool = True,
    ) -> tuple[bool, dict[str, Any]]:
        """Check flow connectivity, enclosed pockets, and XY feature limits.

        Feature checks use 4-connected pixel components and their bounding-box
        thickness as a first-pass manufacturability screen. A final CAD/DfAM
        review is still needed for true 3D overhang angle and local geometry.
        """
        solid = np.asarray(occupancy_grid, dtype=bool)
        errors: list[str] = []
        if solid.shape != (self.grid_ny, self.grid_nx):
            return False, {"errors": [f"Expected grid {(self.grid_ny, self.grid_nx)}, got {solid.shape}"]}
        ny, nx = solid.shape
        min_cells = int(math.ceil(self.min_feature_mm / self.element_size_mm - 1e-10))
        max_cells = int(math.floor(self.max_fin_thickness_mm / self.element_size_mm + 1e-10))

        # Keep the flow opening connected to explicit inlet/outlet sides.
        # Solid on an external edge would also make boundary classification
        # and the left/right boundary conditions ambiguous.
        if solid[:, 0].any() or solid[:, -1].any():
            errors.append("Solid cells touch the inlet or outlet edge")
        fluid_components = GeometryUtilities._run_components(~solid)
        if any(not c["touches_boundary"] for c in fluid_components):
            errors.append("Enclosed fluid pocket detected")
        if not any(c["touches_inlet"] and c["touches_outlet"] for c in fluid_components):
            errors.append("No continuous inlet-to-outlet fluid path")
        solid_fraction = float(solid.mean())
        if enforce_solid_fraction and not self.solid_fraction_min <= solid_fraction <= self.solid_fraction_max:
            errors.append(
                f"Solid fraction {solid_fraction:.3f} is outside target range "
                f"[{self.solid_fraction_min:.3f}, {self.solid_fraction_max:.3f}]"
            )

        solid_components = GeometryUtilities._run_components(solid)
        for component in solid_components:
            xmin, ymin, xmax, ymax = component["bbox"]
            thickness = min(xmax - xmin + 1, ymax - ymin + 1)
            if thickness < min_cells:
                errors.append(f"Solid feature below minimum {self.min_feature_mm:.3f} mm")
                break
            if thickness > max_cells:
                errors.append(f"Fin thickness exceeds maximum {self.max_fin_thickness_mm:.3f} mm")
                break

        # Reject narrow fluid slits trapped between fins in either grid axis.
        for line in list(~solid) + list((~solid).T):
            for start, stop in GeometryUtilities._line_runs(line):
                bounded = start > 0 and stop < len(line)
                if bounded and stop - start < min_cells:
                    errors.append("Fluid clearance below the minimum feature size")
                    break
            if errors and errors[-1] == "Fluid clearance below the minimum feature size":
                break

        return not errors, {
            "valid": not errors,
            "errors": errors,
            "min_feature_cells": min_cells,
            "max_fin_thickness_cells": max_cells,
            "solid_fraction": solid_fraction,
            "target_solid_fraction": self.target_solid_fraction,
            "solid_fraction_tolerance": self.solid_fraction_tolerance,
            "solid_fraction_enforced": bool(enforce_solid_fraction),
            "fluid_components": len(fluid_components),
            "solid_components": len(solid_components),
            "xy_only": True,
            "limitations": ["3D overhang angle is not checked by this 2D generator"],
        }

    def _random_grid(self) -> np.ndarray:
        """Create a random valid layout at the requested solid fraction.

        Sparse and moderate-density layouts use separated random rectangles.
        When rectangle packing stalls, a randomized-column fin fallback
        reliably reaches the requested density while
        preserving a connected inlet-to-outlet fluid route.
        """
        self.last_random_strategy = None
        self.last_random_column_count = 0
        ny, nx = self.grid_ny, self.grid_nx
        min_cells = int(math.ceil(self.min_feature_mm / self.element_size_mm - 1e-10))
        max_cells = int(math.floor(self.max_fin_thickness_mm / self.element_size_mm + 1e-10))
        if min(ny, nx) < 2 * min_cells + 3:
            raise ValueError("Random fin generation needs a larger grid for clearances")

        # Always try scattered rectangles first. If packing stalls, the fallback
        # retains random column placement and fin spacing rather than reverting
        # to uninterrupted full-width rows.
        solid = np.zeros((ny, nx), dtype=bool)
        clearance = min_cells
        occupied = np.zeros_like(solid)
        route_width = min_cells
        route_y0 = max(min_cells, ny // 2 - route_width // 2)
        route_y1 = route_y0 + route_width
        total_cells = nx * ny
        target_cells = math.ceil(self.solid_fraction_min * total_cells)
        maximum_cells = math.floor(self.solid_fraction_max * total_cells)
        # Count placed cells incrementally; recomputing solid.mean() scans the
        # multi-million-cell grid for every candidate and caused hour-long runs.
        attempt_limit = min(
            20_000,
            max(5_000, int(self.solid_fraction_min * total_cells /
                           max_cells**2 * 5)),
        )
        stall_limit = 750
        attempts = 0
        stalled = 0
        solid_cells = 0
        while solid_cells < target_cells and attempts < attempt_limit and stalled < stall_limit:
            attempts += 1
            horizontal = bool(self.rng.random() < 0.8)
            thickness = int(self.rng.integers(min_cells, max_cells + 1))
            long_max = min(nx - 2 * clearance, max(6 * min_cells, 120))
            if long_max < 2 * min_cells:
                break
            length = int(self.rng.integers(2 * min_cells, long_max + 1))
            width_x, width_y = (length, thickness) if horizontal else (thickness, length)
            if width_x > nx - 2 * clearance or width_y > ny - 2 * clearance:
                stalled += 1
                continue
            x0 = int(self.rng.integers(clearance, nx - clearance - width_x + 1))
            y0 = int(self.rng.integers(clearance, ny - clearance - width_y + 1))
            y1 = y0 + width_y
            if y0 < route_y1 + clearance and y1 > route_y0 - clearance:
                stalled += 1
                continue
            ex0, ex1 = max(0, x0 - clearance), min(nx, x0 + width_x + clearance)
            ey0, ey1 = max(0, y0 - clearance), min(ny, y1 + clearance)
            candidate_area = width_x * width_y
            if occupied[ey0:ey1, ex0:ex1].any() or solid_cells + candidate_area > maximum_cells:
                stalled += 1
                continue
            solid[y0:y1, x0:x0 + width_x] = True
            occupied[ey0:ey1, ex0:ex1] = True
            solid_cells += candidate_area
            stalled = 0

        if solid_cells >= target_cells:
            self.last_random_strategy = "random_rectangles"
            return solid
        self.last_random_strategy = "random_columns_varied_fins_fallback"
        print("Random rectangle packing stalled; falling back to randomized columns and gapped fins")
        return self._random_lamella_grid()

    def _random_lamella_grid(self) -> np.ndarray:
        """Fallback with randomized vertical gashes and varied horizontal fins."""
        ny, nx = self.grid_ny, self.grid_nx
        min_cells = int(math.ceil(self.min_feature_mm / self.element_size_mm - 1e-10))
        max_cells = int(math.floor(self.max_fin_thickness_mm / self.element_size_mm + 1e-10))
        x_margin = min_cells
        x_start, x_end = x_margin, nx - x_margin
        active_width = x_end - x_start
        total_cells = nx * ny

        width_span = int(round(self.fallback_column_width_variation * (max_cells - min_cells)))
        max_gash_width = min(max_cells, min_cells + width_span)
        requested_columns = int(self.rng.integers(
            self.fallback_column_count_min, self.fallback_column_count_max + 1
        ))
        gashes = []
        occupied_x = np.zeros(nx, dtype=bool)
        placement_attempts = 0
        while len(gashes) < requested_columns and placement_attempts < requested_columns * 100:
            placement_attempts += 1
            width = int(self.rng.integers(min_cells, max_gash_width + 1))
            gx0 = int(self.rng.integers(x_start, x_end - width + 1))
            exclusion_start = max(x_start, gx0 - min_cells)
            exclusion_end = min(x_end, gx0 + width + min_cells)
            if occupied_x[exclusion_start:exclusion_end].any():
                continue
            occupied_x[gx0:gx0 + width] = True

            if self.rng.random() < self.fallback_full_height_gash_probability:
                gy0, gy1 = 0, ny
            else:
                h_min = max(min_cells, int(math.ceil(self.fallback_partial_gash_min_fraction * ny)))
                h_max = max(h_min, int(math.floor(self.fallback_partial_gash_max_fraction * ny)))
                h_max = min(h_max, ny)
                height = int(self.rng.integers(h_min, h_max + 1))
                gy0 = int(self.rng.integers(0, ny - height + 1))
                gy1 = gy0 + height
            gashes.append((gx0, gx0 + width, gy0, gy1))

        if not gashes:
            raise RuntimeError("Could not place any fallback vertical gashes")
        self.last_random_column_count = len(gashes)

        # Compensate the starting fin coverage for the area removed by gashes.
        # The final density is then checked by the normal Generator validator.
        gash_area = sum((x1 - x0) * (y1 - y0) for x0, x1, y0, y1 in gashes)
        active_fraction = active_width / nx
        removed_fraction_within_active = gash_area / (active_width * ny)
        retained_fraction = max(1e-6, 1.0 - removed_fraction_within_active)
        local_target = self.target_solid_fraction / (active_fraction * retained_fraction)
        desired_rows = int(round(min(local_target, 1.0) * ny))

        max_n_fins = ny // (2 * min_cells)
        feasible_row_plans = []
        search_radius = max(20, int(math.ceil(self.solid_fraction_tolerance * nx / active_fraction)) + 4)
        row_low = max(1, desired_rows - search_radius)
        row_high = min(ny, desired_rows + search_radius)
        for solid_rows in range(row_low, row_high + 1):
            for n_fins in range(1, max_n_fins + 1):
                if (n_fins * min_cells <= solid_rows <= n_fins * max_cells
                        and ny - solid_rows >= (n_fins + 1) * min_cells):
                    feasible_row_plans.append((solid_rows, n_fins))
        if not feasible_row_plans:
            raise ValueError(
                "Could not form the requested fin fraction within thickness and clearance limits. "
                "Adjust the target/tolerance, column settings, or feature limits."
            )
        best_error = min(abs(rows - desired_rows) for rows, _ in feasible_row_plans)
        nearest = [plan for plan in feasible_row_plans if abs(plan[0] - desired_rows) == best_error]
        solid_rows, n_fins_min = nearest[int(self.rng.integers(len(nearest)))]
        feasible_counts = [n for rows, n in feasible_row_plans if rows == solid_rows]
        n_fins_max = max(feasible_counts)
        # Bias toward fewer, thicker fins while allowing a randomized assortment.
        n_fins = int(self.rng.integers(n_fins_min, min(n_fins_max, n_fins_min + max(2, n_fins_min // 3)) + 1))

        thicknesses = np.full(n_fins, min_cells, dtype=int)
        extra_solid = solid_rows - n_fins * min_cells
        capacity = max_cells - min_cells
        if extra_solid and capacity:
            slots = np.repeat(np.arange(n_fins), capacity)
            self.rng.shuffle(slots)
            thicknesses += np.bincount(slots[:extra_solid], minlength=n_fins)
        self.rng.shuffle(thicknesses)

        extra_fluid = ny - solid_rows - (n_fins + 1) * min_cells
        gap_extras = self.rng.multinomial(extra_fluid, np.full(n_fins + 1, 1.0 / (n_fins + 1)))
        gaps = min_cells + gap_extras
        solid = np.zeros((ny, nx), dtype=bool)
        y = int(gaps[0])
        for i, thickness in enumerate(thicknesses):
            solid[y:y + int(thickness), x_start:x_end] = True
            y += int(thickness) + int(gaps[i + 1])

        # Carve the vertical gashes. Some span the full height; the rest occupy
        # a random 5-10% height interval and cut only the fins they intersect.
        for gx0, gx1, gy0, gy1 in gashes:
            solid[gy0:gy1, gx0:gx1] = False
        return solid

    def _programmatic_grid(self) -> np.ndarray:
        """Create an empty channel or evenly spaced horizontal validation fins.

        Programmatic validation geometries intentionally do not have to meet
        the random sampler's target solid-fraction band.
        """
        ny, nx = self.grid_ny, self.grid_nx
        solid = np.zeros((ny, nx), dtype=bool)
        if self.programmatic_layout == "empty":
            return solid

        min_cells = int(math.ceil(self.min_feature_mm / self.element_size_mm - 1e-10))
        max_cells = int(math.floor(self.max_fin_thickness_mm / self.element_size_mm + 1e-10))
        thickness_cells = int(round(self.programmatic_fin_thickness_mm / self.element_size_mm))
        if not min_cells <= thickness_cells <= max_cells:
            raise ValueError(
                "Programmatic fin thickness rounds to a grid width outside the configured feature limits"
            )
        inlet_cells = int(round(self.programmatic_inlet_length_mm / self.element_size_mm))
        outlet_cells = int(round(self.programmatic_outlet_length_mm / self.element_size_mm))
        x0, x1 = inlet_cells, nx - outlet_cells
        if x0 < 0 or outlet_cells < 0 or x1 - x0 < 2 * min_cells:
            raise ValueError("Programmatic inlet/outlet lengths leave too little room for the fins")
        if x0 < min_cells or outlet_cells < min_cells:
            raise ValueError(
                "Programmatic inlet and outlet sections must each be at least the minimum feature size"
            )
        n_fins = self.programmatic_n_fins
        fluid_cells = ny - n_fins * thickness_cells
        if fluid_cells < (n_fins + 1) * min_cells:
            raise ValueError(
                "Programmatic fins leave insufficient minimum clearance; reduce fin count/thickness"
            )
        gap_base, gap_remainder = divmod(fluid_cells, n_fins + 1)
        gaps = [gap_base + (i < gap_remainder) for i in range(n_fins + 1)]
        y = gaps[0]
        for i in range(n_fins):
            solid[y:y + thickness_cells, x0:x1] = True
            y += thickness_cells + gaps[i + 1]
        return solid

    def _neural_grid(self) -> np.ndarray:
        if self.neural_model is None or not self.model_trained:
            raise RuntimeError("Train the CNN with train_stage1() before neural generation")
        latent = self.rng.standard_normal((1, self.latent_dim)).astype(np.float32)
        if hasattr(self.neural_model, "generate_grid"):
            raw = self.neural_model.generate_grid(self.grid_ny, self.grid_nx, latent[0])
            if hasattr(raw, "detach"):
                raw = raw.detach().cpu().numpy()
        else:
            model = self.neural_model
            was_training = getattr(model, "training", None)
            if hasattr(model, "eval"):
                model.eval()
            with torch.no_grad():
                raw = model(torch.from_numpy(latent))
            if was_training and hasattr(model, "train"):
                model.train()
            if hasattr(raw, "detach"):
                raw = raw.detach().cpu().numpy()
        grid = np.asarray(raw, dtype=np.float32).squeeze()
        if grid.shape != (self.grid_ny, self.grid_nx):
            raise ValueError(
                f"neural_model returned {grid.shape}; expected {(self.grid_ny, self.grid_nx)}"
            )
        if not np.isfinite(grid).all():
            raise ValueError("neural_model output contains NaN or infinite values")
        return grid >= 0.5  # model contract: occupancy probabilities, solid-high

    def generate(self, method: str = "random") -> HeatExchanger:
        """Generate one valid HeatExchanger using random, neural, or programmatic geometry."""
        method = method.lower()
        if method not in {"random", "neural", "programmatic"}:
            raise ValueError("method must be 'random', 'neural', or 'programmatic'")
        for _ in range(self.max_attempts):
            if method == "random":
                grid = self._random_grid()
            elif method == "neural":
                grid = self._neural_grid()
            else:
                grid = self._programmatic_grid()
            valid, report = self.validate(
                grid,
                enforce_solid_fraction=(method != "programmatic"),
            )
            self.last_validation_report = report
            if valid:
                print(
                    f"Generated {method} design: solid density={report['solid_fraction']:.1%}"
                )
                if method == "random":
                    print(
                        f"Random strategy: {self.last_random_strategy}; "
                        f"vertical columns/gashes: {self.last_random_column_count}"
                    )
                return HeatExchanger(
                    occupancy_grid=grid,
                    fin_plate_x_mm=self.fin_plate_x_mm,
                    fin_plate_y_mm=self.fin_plate_y_mm,
                    element_size_mm=self.element_size_mm,
                    base_dir=self.base_dir,
                    mesh_generator=self.mesh_generator,
                    metadata={
                        "generation_method": method,
                        "programmatic_layout": self.programmatic_layout if method == "programmatic" else None,
                        "random_strategy": self.last_random_strategy if method == "random" else None,
                        "random_column_count": self.last_random_column_count if method == "random" else None,
                        "validation": report,
                        "chip_gap_mm": self.chip_gap_mm,
                    },
                )
        detail = self.last_validation_report or {}
        raise RuntimeError(f"Could not generate a valid {method} design in {self.max_attempts} attempts: {detail}")

    def generate_batch(self, n_designs: int, method: str = "random") -> list[HeatExchanger]:
        """Generate ``n_designs`` valid HeatExchanger objects."""
        if n_designs < 0:
            raise ValueError("n_designs must be nonnegative")
        return [self.generate(method=method) for _ in range(n_designs)]

In [85]:
class MooseInputWriter:
    """Render a steady Test 1 MOOSE input for fluid flow and prescribed fin-wall temperature."""

    TEST1_TEMPLATE = r"""# Test 1: fluid flow plus temperature in fluid and solid fins.
# No chip heat generation or baseplate coupling is included in this test.
rho = {{ rho }}
mu = {{ mu }}
k = {{ k }}
cp = {{ cp }}
k_solid = {{ k_solid }}

[GlobalParams]
    gravity = '0 0 0'
    pspg = true
[]

[Mesh]
    [gen]
        type = FileMeshGenerator
        file = '{{ mesh_name }}'
    []
[]

[Variables]
    [vel_x]
        block = Fluid
    []
    [vel_y]
        block = Fluid
    []
    [p]
        block = Fluid
    []
    [T_f]
        block = Fluid
    []
    [T_s]
        block = Solid
    []
[]

[Kernels]
    [mass]
        type = INSMass
        variable = p
        pressure = p
        u = vel_x
        v = vel_y
        block = Fluid
    []
    [x_momentum_space]
        type = INSMomentumLaplaceForm
        variable = vel_x
        u = vel_x
        v = vel_y
        pressure = p
        component = 0
        block = Fluid
    []
    [y_momentum_space]
        type = INSMomentumLaplaceForm
        variable = vel_y
        u = vel_x
        v = vel_y
        pressure = p
        component = 1
        block = Fluid
    []
    [fluid_temperature]
        type = INSTemperature
        variable = T_f
        u = vel_x
        v = vel_y
        block = Fluid
    []
    [solid_conduction]
        type = HeatConduction
        variable = T_s
        block = Solid
    []
[]

[BCs]
    [bottom_no_slip_x]
        type = DirichletBC
        variable = vel_x
        boundary = BottomWall
        value = 0
    []
    [bottom_no_slip_y]
        type = DirichletBC
        variable = vel_y
        boundary = BottomWall
        value = 0
    []
    [top_symmetry_normal]
        type = DirichletBC
        variable = vel_y
        boundary = Symmetry
        value = 0
    []
    [top_symmetry_tangent]
        type = NeumannBC
        variable = vel_x
        boundary = Symmetry
        value = 0
    []
    [inlet_u]
        type = FunctionDirichletBC
        variable = vel_x
        boundary = Inlet
        function = inlet_function
    []
    [inlet_v]
        type = DirichletBC
        variable = vel_y
        boundary = Inlet
        value = 0
    []
    [pressure_pin]
        type = DirichletBC
        variable = p
        boundary = Outlet
        value = {{ P_pin }}
    []
    [fluid_wall_temperature]
        type = DirichletBC
        variable = T_f
        boundary = FinWalls
        value = {{ T_interface }}
    []
    [solid_wall_temperature]
        type = DirichletBC
        variable = T_s
        boundary = FinWalls
        value = {{ T_interface }}
    []
    [fluid_inlet_temperature]
        type = DirichletBC
        variable = T_f
        boundary = Inlet
        value = {{ T_in }}
    []
    [fluid_bottom_adiabatic]
        type = NeumannBC
        variable = T_f
        boundary = BottomWall
        value = 0
    []
    [fluid_symmetry_temperature]
        type = NeumannBC
        variable = T_f
        boundary = Symmetry
        value = 0
    []
[]

[ICs]
    [vel_x_ic]
        type = ConstantIC
        variable = vel_x
        value = {{ U_in }}
    []
    [T_f_ic]
        type = ConstantIC
        variable = T_f
        value = {{ T_in }}
    []
    [T_s_ic]
        type = ConstantIC
        variable = T_s
        value = {{ T_interface }}
    []
[]

[Materials]
    [fluid_properties]
        type = GenericConstantMaterial
        block = Fluid
        prop_names = 'rho mu k cp'
        prop_values = '${rho} ${mu} ${k} ${cp}'
    []
    [solid_thermal]
        type = HeatConductionMaterial
        block = Solid
        thermal_conductivity = ${k_solid}
    []
[]

[Functions]
    [inlet_function]
        type = ParsedFunction
        expression = '{{ inlet_expression }}'
    []
[]

[Preconditioning]
    [SMP]
        type = SMP
        full = true
        solve_type = NEWTON
    []
[]

[Executioner]
    type = Steady
    solve_type = NEWTON
    automatic_scaling = true
    compute_scaling_once = false
    off_diagonals_in_auto_scaling = true
    resid_vs_jac_scaling_param = 0
    petsc_options_iname = '-pc_type -pc_factor_mat_solver_type'
    petsc_options_value = 'lu mumps'
    line_search = bt
    nl_rel_tol = 1e-8
    nl_abs_tol = 1e-12
    nl_max_its = 50
[]

[Debug]
    show_var_residual_norms = true
[]

[Outputs]
    exodus = true
    csv = true
[]
"""
    TEST1_AD_TEMPLATE = r"""# Test 1: fluid flow plus temperature in fluid and solid fins.
    # No chip heat generation or baseplate coupling is included in this test.
    rho = {{ rho }}
    mu = {{ mu }}
    k = {{ k }}
    cp = {{ cp }}
    k_solid = {{ k_solid }}
    
    [Mesh]
        [gen]
            type = FileMeshGenerator
            file = '{{ mesh_name }}'
        []
    []
    
    [Variables]
        [velocity]
            family = LAGRANGE_VEC
            order = FIRST
            block = Fluid
        []
        [p]
            family = LAGRANGE
            order = FIRST
            block = Fluid
        []
        [T_f]
            family = LAGRANGE
            order = FIRST
            block = Fluid
        []
        [T_s]
            family = LAGRANGE
            order = FIRST
            block = Solid
        []
    []
    
    [Kernels]
        # Steady incompressible flow, fluid block only
        [mass]
            type = INSADMass
            variable = p
            block = Fluid
        []
        [mass_pspg]
            type = INSADMassPSPG
            variable = p
            block = Fluid
        []
        [momentum_advection]
            type = INSADMomentumAdvection
            variable = velocity
            block = Fluid
        []
        [momentum_viscous]
            type = INSADMomentumViscous
            variable = velocity
            block = Fluid
        []
        [momentum_pressure]
            type = INSADMomentumPressure
            variable = velocity
            pressure = p
            integrate_p_by_parts = true
            block = Fluid
        []
        [momentum_supg]
            type = INSADMomentumSUPG
            variable = velocity
            velocity = velocity
            block = Fluid
        []

        # Steady fluid energy equation
        [fluid_conduction]
            type = ADHeatConduction
            variable = T_f
            thermal_conductivity = k
            block = Fluid
        []
        [fluid_advection]
            type = INSADEnergyAdvection
            variable = T_f
            block = Fluid
        []
        [fluid_energy_supg]
            type = INSADEnergySUPG
            variable = T_f
            velocity = velocity
            block = Fluid
        []

        # Steady solid conduction
        [solid_conduction]
            type = ADHeatConduction
            variable = T_s
            thermal_conductivity = k
            block = Solid
        []
    []
    
    [BCs]
        [bottom_no_slip]
            type = ADVectorFunctionDirichletBC
            variable = velocity
            boundary = BottomWall
            function_x = 0
            function_y = 0
            set_z_comp = false
        []

        [fin_wall_no_slip]
            type = ADVectorFunctionDirichletBC
            variable = velocity
            boundary = FinWalls
            function_x = 0
            function_y = 0
            set_z_comp = false
        []

        [top_symmetry_normal]
            type = ADVectorFunctionDirichletBC
            variable = velocity
            boundary = Symmetry
            function_y = 0
            set_x_comp = false
            set_y_comp = true
            set_z_comp = false
        []

        [inlet_velocity]
            type = ADVectorFunctionDirichletBC
            variable = velocity
            boundary = Inlet
            function_x = inlet_function
            function_y = 0
            set_z_comp = false
        []

        [pressure_pin]
            type = DirichletBC
            variable = p
            boundary = Outlet
            value = {{ P_pin }}
        []

        [fluid_wall_temperature]
            type = DirichletBC
            variable = T_f
            boundary = FinWalls
            value = {{ T_interface }}
        []

        [solid_wall_temperature]
            type = DirichletBC
            variable = T_s
            boundary = FinWalls
            value = {{ T_interface }}
        []

        [fluid_inlet_temperature]
            type = DirichletBC
            variable = T_f
            boundary = Inlet
            value = {{ T_in }}
        []
    []
    
    [ICs]
        [vel_x_ic]
            type = VectorFunctionIC
            variable = velocity
            function_x = inlet_function
            function_y = 0.0
            function_z = 0.0
        []
        [T_f_ic]
            type = ConstantIC
            variable = T_f
            value = {{ T_in }}
        []
        [T_s_ic]
            type = ConstantIC
            variable = T_s
            value = {{ T_interface }}
        []
    []
    
    [Materials]
        [fluid_properties]
            type = ADGenericConstantMaterial
            block = Fluid
            prop_names = 'rho mu k cp'
            prop_values = '${rho} ${mu} ${k} ${cp}'
        []
        [fluid_ins_stabilization]
            type = INSADStabilized3Eqn
            block = Fluid
            velocity = velocity
            pressure = p
            temperature = T_f
        []
        [solid_properties]
            type = ADGenericConstantMaterial
            block = Solid
            prop_names = 'k'
            prop_values = '${k_solid}'
        []
    []
    
    [Functions]
        [inlet_function]
            type = ParsedFunction
            expression = '{{ inlet_expression }}'
        []
    []
    
    [Preconditioning]
        [SMP]
            type = SMP
            full = true
            solve_type = NEWTON
        []
    []
    
    [Executioner]
        type = Steady
        solve_type = NEWTON
        automatic_scaling = true
        compute_scaling_once = false
        off_diagonals_in_auto_scaling = true
        resid_vs_jac_scaling_param = 0
        petsc_options_iname = '-pc_type -pc_factor_mat_solver_type'
        petsc_options_value = 'lu mumps'
        line_search = bt
        nl_rel_tol = 1e-8
        nl_abs_tol = 1e-12
        nl_max_its = 50
    []
    
    [Debug]
        show_var_residual_norms = true
    []
    
    [Outputs]
        exodus = true
        csv = true
    []
    """

    TEST2_TEMPLATE = r"""# Test 2: full physics with chip heat generation and baseplate coupling.
        d_ref = {{ channel_depth_m }}           # m
        t_base = {{ baseplate_thickness_m }}    # m
        k_base = {{ baseplate_conductivity }}   # W/(m*K)
        rho = {{ rho }}                         # kg/m^3
        mu = {{ mu }}                           # Pa*s
        k = {{ k }}                             # W/(m*K) 
        cp = {{ cp }}                           # J/(kg*K)
        k_solid = {{ k_solid }}                 # W/(m*K)
        
        [Mesh]
            [gen]
                type = FileMeshGenerator
                file = '{{ mesh_name }}'
            []
        []
        
        [Variables]
            [velocity]
                family = LAGRANGE_VEC
                order = FIRST
                block = Fluid
            []
            [p]
                family = LAGRANGE
                order = FIRST
                block = Fluid
            []
            [T_f]
                family = LAGRANGE
                order = FIRST
                block = Fluid
            []
            [T_b]
                family = LAGRANGE
                order = FIRST
                block = 'Fluid Solid'
            []
        []
        
        [Kernels]
            # Steady incompressible flow, fluid block only
            [mass]
                type = INSADMass
                variable = p
                block = Fluid
            []
            [mass_pspg]
                type = INSADMassPSPG
                variable = p
                block = Fluid
            []
            [momentum_advection]
                type = INSADMomentumAdvection
                variable = velocity
                block = Fluid
            []
            [momentum_viscous]
                type = INSADMomentumViscous
                variable = velocity
                block = Fluid
            []
            [momentum_pressure]
                type = INSADMomentumPressure
                variable = velocity
                pressure = p
                integrate_p_by_parts = true
                block = Fluid
            []
            [momentum_supg]
                type = INSADMomentumSUPG
                variable = velocity
                velocity = velocity
                block = Fluid
            []
    
            # Steady fluid energy equation
            [fluid_conduction]
                type = ADHeatConduction
                variable = T_f
                thermal_conductivity = k
                block = Fluid
            []
            [fluid_advection]
                type = INSADEnergyAdvection
                variable = T_f
                block = Fluid
            []
            [fluid_energy_supg]
                type = INSADEnergySUPG
                variable = T_f
                velocity = velocity
                block = Fluid
            []
    
            # In-plane conduction in the depth-averaged baseplate.
            [baseplate_conduction]
                type = ADHeatConduction
                variable = T_b
                thermal_conductivity = k_base_2d
                block = 'Fluid Solid'
            []

            # Chip heat enters the baseplate as a surface map. Dividing by d_ref
            # makes it a source consistent with the 2D equations per unit depth.
            [chip_heat]
                type = HeatSource
                variable = T_b
                function = chip_flux
                value = {{ q_chip_source_scale }}
                block = 'Fluid Solid'
            []

            # Equal-and-opposite heat exchange between baseplate and fluid under
            # open fluid areas. Reaction adds +h(T_b or T_f); CoupledForce subtracts
            # the other field.
            [floor_exchange_base_diagonal]
                type = ADReaction
                variable = T_b
                rate = {{ h_floor_over_depth }}
                block = Fluid
            []
            [floor_exchange_base_from_fluid]
                type = ADCoupledForce
                variable = T_b
                v = T_f
                coef = {{ h_floor_over_depth }}
                block = Fluid
            []
            [floor_exchange_fluid_diagonal]
                type = ADReaction
                variable = T_f
                rate = {{ h_floor_over_depth }}
                block = Fluid
            []
            [floor_exchange_fluid_from_base]
                type = ADCoupledForce
                variable = T_f
                v = T_b
                coef = {{ h_floor_over_depth }}
                block = Fluid
            []
        []

        [InterfaceKernels]
            # Replace both fixed FinWalls temperature BCs with this conjugate
            # exchange. T_b supplies the fin-root/base temperature; fin efficiency
            # reduces the heat transfer to account for cooler fin tips.
            [fin_to_fluid_heat]
                type = ConjugateHeatTransfer
                variable = T_f
                T_fluid = T_f
                neighbor_var = T_b
                boundary = FinWalls
                htc = h_fin_effective
            []
        []
        
        [BCs]
            [bottom_no_slip]
                type = ADVectorFunctionDirichletBC
                variable = velocity
                boundary = BottomWall
                function_x = 0
                function_y = 0
                set_z_comp = false
            []
    
            [fin_wall_no_slip]
                type = ADVectorFunctionDirichletBC
                variable = velocity
                boundary = FinWalls
                function_x = 0
                function_y = 0
                set_z_comp = false
            []
    
            [top_symmetry_normal]
                type = ADVectorFunctionDirichletBC
                variable = velocity
                boundary = Symmetry
                function_y = 0
                set_x_comp = false
                set_y_comp = true
                set_z_comp = false
            []
    
            [inlet_velocity]
                type = ADVectorFunctionDirichletBC
                variable = velocity
                boundary = Inlet
                function_x = inlet_function
                function_y = 0
                set_z_comp = false
            []
    
            [pressure_pin]
                type = DirichletBC
                variable = p
                boundary = Outlet
                value = {{ P_pin }}
            []
    
            [fluid_inlet_temperature]
                type = DirichletBC
                variable = T_f
                boundary = Inlet
                value = {{ T_in }}
            []
        []
        
        [ICs]
            [velocity_ic]
                type = VectorFunctionIC
                variable = velocity
                function_x = inlet_function
                function_y = 0.0
                function_z = 0.0
            []
            [T_f_ic]
                type = ConstantIC
                variable = T_f
                value = {{ T_in }}
            []
            [T_b_ic]
                type = ConstantIC
                variable = T_b
                value = {{ T_in }}
            []
        []
        
        [Materials]
            [fluid_properties]
                type = ADGenericConstantMaterial
                block = Fluid
                prop_names = 'rho mu k cp'
                prop_values = '${rho} ${mu} ${k} ${cp}'
            []
            [fluid_ins_stabilization]
                type = INSADStabilized3Eqn
                block = Fluid
                velocity = velocity
                pressure = p
                temperature = T_f
            []
            [baseplate_properties]
                type = ADGenericConstantMaterial
                block = 'Fluid Solid'
                prop_names = 'k_base_2d'
                prop_values = '{{k_base_2d }}'
            []
            [fin_heat_transfer]
                type = ADGenericConstantMaterial
                block = Fluid
                prop_names = 'h_fin_effective'
                prop_values = '{{ h_fin_effective }}'
            []
        []
        
        [Functions]
            [inlet_function]
                type = ParsedFunction
                expression = '{{ inlet_expression }}'
            []
            [chip_flux]
                type = PiecewiseMulticonstant
                data_file = '{{ chip_power_map_file }}'
                direction = 'right right'
            []
        []
        
        [Preconditioning]
            [SMP]
                type = SMP
                full = true
                solve_type = NEWTON
            []
        []
        
        [Executioner]
            type = Steady
            solve_type = NEWTON
            automatic_scaling = true
            compute_scaling_once = false
            off_diagonals_in_auto_scaling = true
            resid_vs_jac_scaling_param = 0
            petsc_options_iname = '-pc_type -pc_factor_mat_solver_type'
            petsc_options_value = 'lu mumps'
            line_search = bt
            nl_rel_tol = 1e-8
            nl_abs_tol = 1e-12
            nl_max_its = 50
        []
        
        [Debug]
            show_var_residual_norms = true
        []
        
        [Outputs]
            exodus = true
            csv = true
        []
        """

    def __init__(
        self,
        fluid_properties: dict[str, float] | None = None,
        solid_properties: dict[str, float] | None = None,
        boundary_conditions: dict[str, Any] | None = None,
        test2_parameters: dict[str, Any] | None = None,
    ):
        self.fluid_properties = fluid_properties or {
            "rho": 997.0, "mu": 8.9e-4, "k": 0.6, "cp": 4180.0,
        }
        self.solid_properties = solid_properties or {"k": 390.0}
        self.boundary_conditions = boundary_conditions or {
            "U_in": self.calculate_uniform_inlet_velocity(),
            "T_in": 293.15, "T_wall": 303.15,
            "P_pin": 0.0, "U_profile_type": "convergent",
            "boundary_layer_fraction": 0.10,
        }
        self.test2_parameters = {
            "channel_depth_m": 0.002,
            "baseplate_thickness_m": 0.002,
            "baseplate_conductivity": 390.0,
            "h_floor_w_m2k": None,
            "h_fin_w_m2k": None,
            "fin_efficiency": None,
            "chip_power_map_file": None,
        }
        self.test2_parameters.update(test2_parameters or {})

    @staticmethod
    def calculate_uniform_inlet_velocity(
        total_flow_lpm: float = 2.25,
        flow_fraction: float = 0.5,
        inlet_height_mm: float = 54.1,
        channel_depth_mm: float = 2.0,
    ) -> float:
        """Calculate mean speed from total flow and one side's inlet area."""
        if total_flow_lpm <= 0 or not 0 < flow_fraction <= 1:
            raise ValueError("Flow rate must be positive and flow_fraction in (0, 1]")
        if inlet_height_mm <= 0 or channel_depth_mm <= 0:
            raise ValueError("Inlet height and channel depth must be positive")
        flow_per_side_m3_s = total_flow_lpm * 1e-3 / 60.0 * flow_fraction
        area_m2 = (inlet_height_mm / 1000.0) * (channel_depth_mm / 1000.0)
        return flow_per_side_m3_s / area_m2

    @staticmethod
    def calculate_plate_fin_efficiency(
        h_w_m2k: float,
        k_w_mk: float,
        thickness_mm: float,
        height_mm: float,
    ) -> float:
        """Estimate straight-plate fin efficiency with an adiabatic tip.

        This is a reduced-order estimate; use the 3D validation to calibrate it.
        """
        if h_w_m2k <= 0 or k_w_mk <= 0 or thickness_mm <= 0 or height_mm <= 0:
            raise ValueError("Fin HTC, conductivity, thickness, and height must be positive")
        thickness_m = thickness_mm / 1000.0
        height_m = height_mm / 1000.0
        m = math.sqrt(2.0 * h_w_m2k / (k_w_mk * thickness_m))
        mL = m * height_m
        return math.tanh(mL) / mL

    def create_input_file(
        self,
        mesh_name: str | Path,
        output_dir: str | Path,
        fluid_properties: dict[str, float] | None = None,
        boundary_conditions: dict[str, Any] | None = None,
        solid_properties: dict[str, float] | None = None,
        template: str = "TEST1",
    ) -> Path:
        """Render a Test 1 or Test 2 MOOSE input file and return its absolute path."""
        mesh_path = Path(mesh_name).expanduser().resolve()
        output_dir = Path(output_dir).expanduser()
        output_dir.mkdir(parents=True, exist_ok=True)
        fluid = {**self.fluid_properties, **(fluid_properties or {})}
        solid = {**self.solid_properties, **(solid_properties or {})}
        bc = {**self.boundary_conditions, **(boundary_conditions or {})}
        required = ("U_in", "T_in", "P_pin")
        missing = [name for name in required if name not in bc]
        if missing:
            raise ValueError(f"Missing boundary-condition values: {missing}")
        if template not in {"TEST1", "TEST1_AD", "TEST2"}:
            raise ValueError("template must be 'TEST1', 'TEST1_AD', or 'TEST2'")
        T_interface = bc.get("T_interface", bc.get("T_wall", bc.get("T_obstructions")))
        if template in {"TEST1", "TEST1_AD"} and T_interface is None:
            raise ValueError("Set boundary_conditions['T_interface'] or ['T_wall']")

        U = float(bc["U_in"])
        profile_type = bc.get("U_profile_type", "uniform").lower()
        if profile_type == "uniform":
            inlet_expression = f"{U:.10g}"
        elif profile_type == "parabolic":
            height = float(bc.get("domain_height", 0.0))
            if height <= 0:
                raise ValueError("domain_height in meters is required for a parabolic inlet")
            inlet_expression = f"{6.0 * U / height**2:.10g}*y*({height:.10g}-y)"
        elif profile_type == "convergent":
            height = float(bc.get("domain_height", 0.0))
            fraction = float(bc.get("boundary_layer_fraction", 0.10))
            if height <= 0:
                raise ValueError("domain_height in meters is required for a convergent inlet")
            if not 0 < fraction <= 1:
                raise ValueError("boundary_layer_fraction must be in (0, 1]")
            delta = fraction * height
            # Start at zero on the bottom wall, rise with a half-parabola, then
            # remain at U up to the top centerline. The derivative is zero at
            # y=delta, so the transition joins the uniform section smoothly.
            inlet_expression = (
                f"if(y <= {delta:.10g}, "
                f"{U:.10g}*(2*y/{delta:.10g}-(y/{delta:.10g})^2), "
                f"{U:.10g})"
            )
        else:
            raise ValueError(
                "U_profile_type must be 'uniform', 'parabolic', or 'convergent'"
            )

        required_fluid = ("rho", "mu", "k", "cp")
        if any(name not in fluid for name in required_fluid) or "k" not in solid:
            raise ValueError("Fluid properties need rho, mu, k, cp; solid properties need k")
        case_id = mesh_path.stem
        if template == "TEST1":
            rendered = Template(self.TEST1_TEMPLATE).render(
                mesh_name=str(mesh_path), case_name=case_id,
                rho=fluid["rho"], mu=fluid["mu"], k=fluid["k"], cp=fluid["cp"],
                k_solid=solid["k"], U_in=U, T_in=bc["T_in"],
                T_interface=T_interface, P_pin=bc["P_pin"],
                inlet_expression=inlet_expression,
            )
        elif template == "TEST1_AD":
            rendered = Template(self.TEST1_AD_TEMPLATE).render(
                mesh_name=str(mesh_path), case_name=case_id,
                rho=fluid["rho"], mu=fluid["mu"], k=fluid["k"], cp=fluid["cp"],
                k_solid=solid["k"], U_in=U, T_in=bc["T_in"],
                T_interface=T_interface, P_pin=bc["P_pin"],
                inlet_expression=inlet_expression,
            )
        elif template == "TEST2":
            t2 = self.test2_parameters
            needed = (
                "channel_depth_m", "baseplate_thickness_m", "baseplate_conductivity",
                "h_floor_w_m2k", "h_fin_w_m2k", "fin_efficiency", "chip_power_map_file",
            )
            missing_t2 = [key for key in needed if t2.get(key) is None]
            if missing_t2:
                raise ValueError(f"Missing Test 2 parameters: {missing_t2}")
            depth = float(t2["channel_depth_m"])
            base_thickness = float(t2["baseplate_thickness_m"])
            k_base = float(t2["baseplate_conductivity"])
            h_floor = float(t2["h_floor_w_m2k"])
            h_fin = float(t2["h_fin_w_m2k"])
            fin_eta = float(t2["fin_efficiency"])
            if depth <= 0 or base_thickness <= 0 or k_base <= 0:
                raise ValueError("Test 2 depth, baseplate thickness, and conductivity must be positive")
            if h_floor <= 0 or h_fin <= 0 or not 0.0 < fin_eta <= 1.0:
                raise ValueError("Test 2 heat-transfer coefficients must be positive and fin_efficiency in (0, 1]")
            chip_map_path = Path(t2["chip_power_map_file"]).expanduser().resolve()
            if not chip_map_path.is_file():
                raise FileNotFoundError(f"Test 2 chip power map does not exist: {chip_map_path}")
            rendered = Template(self.TEST2_TEMPLATE).render(
                mesh_name=str(mesh_path), case_name=case_id,
                rho=fluid["rho"], mu=fluid["mu"], k=fluid["k"], cp=fluid["cp"],
                k_solid=solid["k"], U_in=U, T_in=bc["T_in"], P_pin=bc["P_pin"],
                inlet_expression=inlet_expression,
                channel_depth_m=depth,
                baseplate_thickness_m=base_thickness,
                baseplate_conductivity=k_base,
                k_base_2d=k_base * base_thickness / depth,
                q_chip_source_scale=1.0 / depth,
                h_floor_over_depth=h_floor / depth,
                h_fin_effective=h_fin * fin_eta,
                chip_power_map_file=str(chip_map_path),
            )
        output_path = (output_dir / f"{case_id}.i").resolve()
        output_path.write_text(rendered)
        return output_path


In [86]:
class ChipPowerMap:
    """Convert the competition's chip power-density CSV into a MOOSE map.

    The CSV uses blank rows/columns as visual chip separators. Those separators
    are treated as gaps of ``chip_gap_mm``; numeric zeroes remain real chip cells.
    The exported file describes the lower-left quarter in local plate coordinates
    (origin at the full plate's lower-left corner), with edge-aligned, piecewise-
    constant values in W/m^2 for MOOSE's PiecewiseMulticonstant function.
    """

    def __init__(
        self,
        csv_path: str | Path = "q-map-wcm2.csv",
        output_path: str | Path = "q_map_LL.dat",
        cell_size_mm: float = 2.0,
        chip_gap_mm: float = 0.0,
        full_domain_x_mm: float = 54.1,
        full_domain_y_mm: float = 52.15,
        hbm_shape_cells: tuple[int, int] = (5, 6),
        gpu_shape_cells: tuple[int, int] = (15, 13),
        expected_hbm_count: int = 8,
        expected_gpu_count: int = 2,
        expected_full_power_w: float | None = 1492.8,
    ):
        self.csv_path = Path(csv_path).expanduser()
        self.output_path = Path(output_path).expanduser()
        self.cell_size_mm = float(cell_size_mm)
        self.chip_gap_mm = float(chip_gap_mm)
        self.full_domain_x_mm = float(full_domain_x_mm)
        self.full_domain_y_mm = float(full_domain_y_mm)
        self.hbm_shape_cells = tuple(hbm_shape_cells)
        self.gpu_shape_cells = tuple(gpu_shape_cells)
        self.expected_hbm_count = int(expected_hbm_count)
        self.expected_gpu_count = int(expected_gpu_count)
        self.expected_full_power_w = expected_full_power_w
        if self.cell_size_mm <= 0 or self.chip_gap_mm < 0:
            raise ValueError("cell_size_mm must be positive and chip_gap_mm nonnegative")
        if self.full_domain_x_mm <= 0 or self.full_domain_y_mm <= 0:
            raise ValueError("Full domain dimensions must be positive")

    def _read_cells(self):
        """Read numeric cells; blank CSV entries become separators (None)."""
        with self.csv_path.open("r", newline="", encoding="utf-8-sig") as stream:
            rows = list(csv.reader(stream))
        if not rows or len({len(row) for row in rows}) != 1:
            raise ValueError("Power-map CSV must be a nonempty rectangular grid")
        cells = []
        for row_number, row in enumerate(rows, start=1):
            parsed = []
            for column_number, raw in enumerate(row, start=1):
                value = raw.strip()
                if not value:
                    parsed.append(None)
                else:
                    try:
                        number = float(value)
                    except ValueError as exc:
                        raise ValueError(
                            f"Nonnumeric CSV value at row {row_number}, column {column_number}: {raw!r}"
                        ) from exc
                    if number < 0:
                        raise ValueError("Power density values cannot be negative")
                    parsed.append(number)
            cells.append(parsed)
        return cells

    @staticmethod
    def _components(cells):
        """Find connected numeric chip rectangles; numeric zeroes are included."""
        nrows, ncols = len(cells), len(cells[0])
        visited = set()
        components = []
        for row in range(nrows):
            for col in range(ncols):
                if cells[row][col] is None or (row, col) in visited:
                    continue
                stack = [(row, col)]
                visited.add((row, col))
                points = []
                while stack:
                    r, c = stack.pop()
                    points.append((r, c))
                    for neighbor in ((r - 1, c), (r + 1, c), (r, c - 1), (r, c + 1)):
                        rr, cc = neighbor
                        if (0 <= rr < nrows and 0 <= cc < ncols
                                and cells[rr][cc] is not None
                                and neighbor not in visited):
                            visited.add(neighbor)
                            stack.append(neighbor)
                r0 = min(r for r, _ in points)
                r1 = max(r for r, _ in points) + 1
                c0 = min(c for _, c in points)
                c1 = max(c for _, c in points) + 1
                if len(points) != (r1 - r0) * (c1 - c0):
                    raise ValueError("Each chip region in the CSV must be a filled rectangle")
                components.append({"r0": r0, "r1": r1, "c0": c0, "c1": c1})
        return sorted(components, key=lambda item: (item["r0"], item["c0"]))

    def _validated_chip_groups(self, cells):
        components = self._components(cells)
        hbm_h, hbm_w = self.hbm_shape_cells
        gpu_h, gpu_w = self.gpu_shape_cells
        hbms = [c for c in components if (c["r1"] - c["r0"], c["c1"] - c["c0"]) == (hbm_h, hbm_w)]
        gpus = [c for c in components if (c["r1"] - c["r0"], c["c1"] - c["c0"]) == (gpu_h, gpu_w)]
        if len(hbms) != self.expected_hbm_count or len(gpus) != self.expected_gpu_count:
            shapes = sorted((c["r1"] - c["r0"], c["c1"] - c["c0"]) for c in components)
            raise ValueError(
                "Unexpected power-map chip layout. Expected "
                f"{self.expected_hbm_count} HBM chips of {hbm_h}x{hbm_w} cells and "
                f"{self.expected_gpu_count} GPUs of {gpu_h}x{gpu_w} cells; found {shapes}."
            )
        gpu_rows = {(g["r0"], g["r1"]) for g in gpus}
        if len(gpu_rows) != 1:
            raise ValueError("The two GPU regions must occupy the same CSV rows")
        gpu_row0, gpu_row1 = next(iter(gpu_rows))
        top_hbm = [h for h in hbms if h["r1"] <= gpu_row0]
        bottom_hbm = [h for h in hbms if h["r0"] >= gpu_row1]
        if len(top_hbm) != 4 or len(bottom_hbm) != 4:
            raise ValueError("Expected four HBM chips above and four below the GPUs")
        return sorted(gpus, key=lambda c: c["c0"]), sorted(bottom_hbm, key=lambda c: c["c0"]), components

    def _chip_rectangles(self, cells, gpus, bottom_hbm):
        """Place chip cells on a centered, zero-gap-capable physical layout."""
        cx = self.full_domain_x_mm / 2.0
        cy = self.full_domain_y_mm / 2.0
        s = self.cell_size_mm
        gap = self.chip_gap_mm
        gpu_h, gpu_w = self.gpu_shape_cells
        hbm_h, hbm_w = self.hbm_shape_cells
        gpu_width = gpu_w * s
        gpu_height = gpu_h * s
        hbm_width = hbm_w * s
        hbm_height = hbm_h * s
        gpu_pair_width = 2 * gpu_width + gap
        hbm_bank_width = 4 * hbm_width + 3 * gap
        active_height = 2 * hbm_height + gpu_height + 2 * gap
        if gpu_pair_width > self.full_domain_x_mm or active_height > self.full_domain_y_mm:
            raise ValueError(
                "The chip layout does not fit the configured full-domain dimensions; "
                "check orientation, chip_gap_mm, and plate dimensions."
            )

        gpu_x0 = cx - gpu_pair_width / 2.0
        gpu_y0 = cy - gpu_height / 2.0
        rectangles = []
        # Only the left GPU can contribute to the lower-left quadrant. Build both
        # first so the symmetry placement is explicit and remains gap-aware.
        for gpu_index, component in enumerate(gpus):
            chip_x0 = gpu_x0 + gpu_index * (gpu_width + gap)
            for local_r, row in enumerate(range(component["r0"], component["r1"])):
                y1 = gpu_y0 + gpu_height - local_r * s
                y0 = y1 - s
                for local_c, col in enumerate(range(component["c0"], component["c1"])):
                    x0 = chip_x0 + local_c * s
                    x1 = x0 + s
                    rectangles.append((x0, x1, y0, y1, cells[row][col] * 10_000.0))

        hbm_x0 = cx - hbm_bank_width / 2.0
        bottom_bank_top = gpu_y0 - gap
        for chip_index, component in enumerate(bottom_hbm):
            chip_x0 = hbm_x0 + chip_index * (hbm_width + gap)
            for local_r, row in enumerate(range(component["r0"], component["r1"])):
                y1 = bottom_bank_top - local_r * s
                y0 = y1 - s
                for local_c, col in enumerate(range(component["c0"], component["c1"])):
                    x0 = chip_x0 + local_c * s
                    x1 = x0 + s
                    rectangles.append((x0, x1, y0, y1, cells[row][col] * 10_000.0))

        # Clip every rectangle to the lower-left quarter. This automatically
        # keeps half of the GPU's center row when the horizontal mirror cuts it.
        clipped = []
        for x0, x1, y0, y1, q_w_m2 in rectangles:
            xa, xb = max(0.0, x0), min(cx, x1)
            ya, yb = max(0.0, y0), min(cy, y1)
            if xb - xa > 1e-12 and yb - ya > 1e-12:
                clipped.append((xa, xb, ya, yb, q_w_m2))
        return clipped, (cx, cy)

    def export_moose(self) -> dict[str, Any]:
        """Write a PiecewiseMulticonstant data file and return a power audit."""
        cells = self._read_cells()
        gpus, bottom_hbm, components = self._validated_chip_groups(cells)
        cell_area_cm2 = (self.cell_size_mm / 10.0) ** 2
        full_power_w = sum(
            value * cell_area_cm2
            for row in cells for value in row if value is not None
        )
        if self.expected_full_power_w is not None and not math.isclose(
            full_power_w, self.expected_full_power_w, rel_tol=0.0, abs_tol=1e-6
        ):
            raise ValueError(
                f"CSV integrates to {full_power_w:.6g} W, expected "
                f"{self.expected_full_power_w:.6g} W. Check units and cell size."
            )

        rectangles, (qx_mm, qy_mm) = self._chip_rectangles(cells, gpus, bottom_hbm)
        # Chip boundaries are built from several arithmetic paths (GPU center,
        # cell offsets, and quadrant clipping). Canonicalize them in mm before
        # deduplicating so floating-point noise cannot create duplicate axis
        # values after conversion to meters for MOOSE.
        axis_precision_mm = 9  # 1e-9 mm is far below the physical grid scale.
        x_edges = sorted({
            round(float(v), axis_precision_mm)
            for v in (0.0, qx_mm, *(v for r in rectangles for v in r[:2]))
        })
        y_edges = sorted({
            round(float(v), axis_precision_mm)
            for v in (0.0, qy_mm, *(v for r in rectangles for v in r[2:4]))
        })
        if any(b <= a for a, b in zip(x_edges, x_edges[1:])):
            raise ValueError("Chip heat-map X axis must be strictly increasing")
        if any(b <= a for a, b in zip(y_edges, y_edges[1:])):
            raise ValueError("Chip heat-map Y axis must be strictly increasing")
        nx, ny = len(x_edges) - 1, len(y_edges) - 1
        bins = np.zeros((ny, nx), dtype=float)
        filled = np.zeros((ny, nx), dtype=bool)
        x_lookup = {value: i for i, value in enumerate(x_edges)}
        y_lookup = {value: i for i, value in enumerate(y_edges)}
        quadrant_power_w = 0.0
        for x0, x1, y0, y1, q_w_m2 in rectangles:
            i0, i1 = x_lookup[round(float(x0), axis_precision_mm)], x_lookup[round(float(x1), axis_precision_mm)]
            j0, j1 = y_lookup[round(float(y0), axis_precision_mm)], y_lookup[round(float(y1), axis_precision_mm)]
            if filled[j0:j1, i0:i1].any():
                raise ValueError("Placed chip cells overlap in the lower-left quadrant")
            bins[j0:j1, i0:i1] = q_w_m2
            filled[j0:j1, i0:i1] = True
            quadrant_power_w += q_w_m2 * ((x1 - x0) / 1000.0) * ((y1 - y0) / 1000.0)

        # PiecewiseMulticonstant with direction='right right' selects the value
        # at the next grid point. Put each rectangle's value at its upper-right
        # edge node so jumps occur exactly on cell boundaries.
        node_values = np.zeros((len(y_edges), len(x_edges)), dtype=float)
        node_values[1:, 1:] = bins
        x_m = [value / 1000.0 for value in x_edges]
        y_m = [value / 1000.0 for value in y_edges]
        self.output_path.parent.mkdir(parents=True, exist_ok=True)
        with self.output_path.open("w", encoding="utf-8", newline="\n") as stream:
            stream.write("# Lower-left quadrant chip heat flux, W/m^2\n")
            stream.write("# Coordinates are in meters; data are ordered with x varying fastest.\n")
            stream.write("AXIS X\n")
            stream.write(" ".join(f"{v:.15g}" for v in x_m) + "\n")
            stream.write("AXIS Y\n")
            stream.write(" ".join(f"{v:.15g}" for v in y_m) + "\n")
            stream.write("DATA\n")
            for row in node_values:
                stream.write(" ".join(f"{v:.12g}" for v in row) + "\n")

        return {
            "output_path": self.output_path.resolve(),
            "csv_rows": len(cells),
            "csv_columns": len(cells[0]),
            "chip_components": len(components),
            "hbm_cell_shape": self.hbm_shape_cells,
            "gpu_cell_shape": self.gpu_shape_cells,
            "full_power_w": full_power_w,
            "lower_left_quadrant_power_w": quadrant_power_w,
            "quadrant_dimensions_mm": (qx_mm, qy_mm),
            "map_grid_nodes": (len(x_edges), len(y_edges)),
        }


In [87]:
# Use the selected accelerator when a neural model is supplied later.
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

In [88]:
# -----------------------------------------------------------------------------
# Competition settings: edit values here, then validate and instantiate.
# Constructor defaults above document the same settings for reuse elsewhere.
# -----------------------------------------------------------------------------
FIN_PLATE_X_MM = 52.15
FIN_PLATE_Y_MM = 54.1
GRID_RESOLUTION = "fine_33um"  # Choose "coarse_100um" or "fine_33um".
ELEMENT_SIZE_OPTIONS_UM = {
    "coarse_100um": 100.0,
    "fine_33um": 100.0 / 3.0,
}
if GRID_RESOLUTION not in ELEMENT_SIZE_OPTIONS_UM:
    raise ValueError(f"Unknown GRID_RESOLUTION: {GRID_RESOLUTION}")
ELEMENT_SIZE_UM = ELEMENT_SIZE_OPTIONS_UM[GRID_RESOLUTION]
MIN_FEATURE_MM = 0.100
MAX_FIN_THICKNESS_MM = 0.500
TARGET_SOLID_FRACTION = 0.40
SOLID_FRACTION_TOLERANCE = 0.10
CHIP_GAP_MM = None  # Fill in when the GPU-to-HBM spacing is known.
BASE_DIR = Path(".")
RANDOM_SEED = 42
GENERATION_METHOD = "programmatic"  # "random", "neural", or "programmatic"
PROGRAMMATIC_LAYOUT = "parallel_fins"  # "empty" or "parallel_fins"; used for programmatic generation.
PROGRAMMATIC_N_FINS = 5
PROGRAMMATIC_FIN_THICKNESS_MM = MIN_FEATURE_MM
PROGRAMMATIC_INLET_LENGTH_MM = 5.0
PROGRAMMATIC_OUTLET_LENGTH_MM = 5.0
FALLBACK_COLUMN_COUNT_MIN = 12
FALLBACK_COLUMN_COUNT_MAX = 24
FALLBACK_COLUMN_WIDTH_VARIATION = 0.70  # 0 = minimum width only; 1 = full configured fin-width span.
FALLBACK_FULL_HEIGHT_GASH_PROBABILITY = 0.25
FALLBACK_PARTIAL_GASH_MIN_FRACTION = 0.05
FALLBACK_PARTIAL_GASH_MAX_FRACTION = 0.10
N_DESIGNS = 1
LATENT_DIM = 32
MAX_GENERATION_ATTEMPTS = 100
MODEL_BASE_CHANNELS = 8
TRAIN_STAGE1 = False
STAGE1_SAMPLES = 16
STAGE1_EPOCHS = 20
STAGE1_BATCH_SIZE = 1
STAGE1_LEARNING_RATE = 1e-3
STAGE1_BETA = 1e-3
NEURAL_MODEL = None  # Optionally supply a trained model.
NEURAL_MODEL_TRAINED = False

MESH_SIZE_MIN_UM = 100.0
MESH_SIZE_MAX_UM = 500.0
MESH_REFINEMENT_DISTANCE_MM = None
REFINE_OUTER_WALLS = False
MESH_VERBOSE = False
MOOSE_OUTPUT_DIR = BASE_DIR / "moose_inputs"
FLUID_PROPERTIES = {"rho": 997.0, "mu": 8.9e-4, "k": 0.6, "cp": 4180.0}
SOLID_PROPERTIES = {"k": 390.0}  # Copper thermal conductivity, W/(m K)
TOTAL_FLOW_RATE_LPM = 2.25
FLOW_FRACTION_PER_SIDE = 0.5
INLET_HEIGHT_MM = 54.1
CHANNEL_DEPTH_MM = 2.0
U_INLET_M_S = MooseInputWriter.calculate_uniform_inlet_velocity(
    total_flow_lpm=TOTAL_FLOW_RATE_LPM,
    flow_fraction=FLOW_FRACTION_PER_SIDE,
    inlet_height_mm=INLET_HEIGHT_MM,
    channel_depth_mm=CHANNEL_DEPTH_MM,
)
TEST1_BOUNDARY_CONDITIONS = {
    "U_in": 0.18,     # Uniform-flow equivalent / post-transition speed, m/s
    "U_profile_type": "convergent",
    "boundary_layer_fraction": 0.15,
    "T_in": 293.15,           # K
    "T_wall": 303.15,         # Prescribed solid-fluid interface temperature, K
    "P_pin": 0.0,             # Pa gauge
}

assert FIN_PLATE_X_MM > 0 and FIN_PLATE_Y_MM > 0
assert ELEMENT_SIZE_UM > 0
assert 0 < MIN_FEATURE_MM <= MAX_FIN_THICKNESS_MM
assert 0.0 <= TARGET_SOLID_FRACTION <= 1.0
assert 0.0 <= SOLID_FRACTION_TOLERANCE <= min(TARGET_SOLID_FRACTION, 1.0 - TARGET_SOLID_FRACTION)
assert CHIP_GAP_MM is None or CHIP_GAP_MM >= 0
assert GENERATION_METHOD in {"random", "neural", "programmatic"}
assert PROGRAMMATIC_LAYOUT in {"empty", "parallel_fins"}
assert N_DESIGNS >= 0

mesh_generator = MeshGenerator(
    element_size_um=ELEMENT_SIZE_UM,
    size_min_um=MESH_SIZE_MIN_UM,
    size_max_um=MESH_SIZE_MAX_UM,
    refine_dist_mm=MESH_REFINEMENT_DISTANCE_MM,
    refine_outer_walls=REFINE_OUTER_WALLS,
    verbose=MESH_VERBOSE,
)
generator = Generator(
    fin_plate_x_mm=FIN_PLATE_X_MM,
    fin_plate_y_mm=FIN_PLATE_Y_MM,
    element_size_um=ELEMENT_SIZE_UM,
    neural_model=NEURAL_MODEL,
    latent_dim=LATENT_DIM,
    model_base_channels=MODEL_BASE_CHANNELS,
    device=device,
    neural_model_trained=NEURAL_MODEL_TRAINED,
    max_attempts=MAX_GENERATION_ATTEMPTS,
    min_feature_mm=MIN_FEATURE_MM,
    max_fin_thickness_mm=MAX_FIN_THICKNESS_MM,
    target_solid_fraction=TARGET_SOLID_FRACTION,
    solid_fraction_tolerance=SOLID_FRACTION_TOLERANCE,
    chip_gap_mm=CHIP_GAP_MM,
    seed=RANDOM_SEED,
    base_dir=BASE_DIR,
    mesh_generator=mesh_generator,
    programmatic_layout=PROGRAMMATIC_LAYOUT,
    programmatic_n_fins=PROGRAMMATIC_N_FINS,
    programmatic_fin_thickness_mm=PROGRAMMATIC_FIN_THICKNESS_MM,
    programmatic_inlet_length_mm=PROGRAMMATIC_INLET_LENGTH_MM,
    programmatic_outlet_length_mm=PROGRAMMATIC_OUTLET_LENGTH_MM,
    fallback_column_count_min=FALLBACK_COLUMN_COUNT_MIN,
    fallback_column_count_max=FALLBACK_COLUMN_COUNT_MAX,
    fallback_column_width_variation=FALLBACK_COLUMN_WIDTH_VARIATION,
    fallback_full_height_gash_probability=FALLBACK_FULL_HEIGHT_GASH_PROBABILITY,
    fallback_partial_gash_min_fraction=FALLBACK_PARTIAL_GASH_MIN_FRACTION,
    fallback_partial_gash_max_fraction=FALLBACK_PARTIAL_GASH_MAX_FRACTION,
)

# Stage 1 learns the valid-geometry prior from the same checked random sampler.
# Stage 2 reinforcement learning for thermal/pressure optimization will be added later.
if TRAIN_STAGE1:
    stage1_history = generator.train_stage1(
        n_samples=STAGE1_SAMPLES,
        num_epochs=STAGE1_EPOCHS,
        batch_size=STAGE1_BATCH_SIZE,
        learning_rate=STAGE1_LEARNING_RATE,
        beta=STAGE1_BETA,
    )

# Generate the selected geometry, mesh both fluid and solid regions, and
# render the MOOSE input. Programmatic layouts intentionally bypass the random
# target-solid-fraction band but still pass connectivity and feature checks.
design = generator.generate(method=GENERATION_METHOD)
mesh_file = design.generate_mesh()
moose_writer = MooseInputWriter(
    fluid_properties=FLUID_PROPERTIES,
    solid_properties=SOLID_PROPERTIES,
    boundary_conditions=TEST1_BOUNDARY_CONDITIONS,
)
input_file = moose_writer.create_input_file(
    mesh_name=mesh_file,
    output_dir=MOOSE_OUTPUT_DIR,
    boundary_conditions={"domain_height": design.actual_dimensions_mm[1] / 1000.0},
    template="TEST1_AD",
)
print(f"Resolved MOOSE input file: {input_file.resolve()}")


Generated programmatic design: solid density=0.7%
Resolved MOOSE input file: /Users/jagardiner/projects/cutthroat/moose_inputs/f373692767ef.i


In [89]:
# -----------------------------------------------------------------------------
# Test 2 input: crop the power map to the lower-left symmetry quadrant and
# export MOOSE PiecewiseMulticonstant data. The mesh used with this map must be
# the corresponding lower-left quadrant, with the same origin and dimensions.
# -----------------------------------------------------------------------------
HEAT_MAP_CSV_PATH = Path("/Users/jagardiner/projects/cutthroat/q-map-wcm2.csv")
HEAT_MAP_OUTPUT_PATH = BASE_DIR / "moose_inputs" / "q_map_LL.dat"
HEAT_MAP_CELL_SIZE_MM = 2.0
HEAT_MAP_CHIP_GAP_MM = 0.0  # Placeholder; blank CSV separators have zero width.
HEAT_MAP_EXPECTED_POWER_W = 1492.8

chip_power_map = ChipPowerMap(
    csv_path=HEAT_MAP_CSV_PATH,
    output_path=HEAT_MAP_OUTPUT_PATH,
    cell_size_mm=HEAT_MAP_CELL_SIZE_MM,
    chip_gap_mm=HEAT_MAP_CHIP_GAP_MM,
    full_domain_x_mm=FIN_PLATE_X_MM,
    full_domain_y_mm=FIN_PLATE_Y_MM,
    hbm_shape_cells=(5, 6),  # 12 x 10 mm from the competition geometry.
    gpu_shape_cells=(15, 13),
    expected_full_power_w=HEAT_MAP_EXPECTED_POWER_W,
)
heat_map_summary = chip_power_map.export_moose()
print(f"CSV total power: {heat_map_summary['full_power_w']:.3f} W")
print(
    "Lower-left quadrant power: "
    f"{heat_map_summary['lower_left_quadrant_power_w']:.3f} W"
)
print(f"Quadrant map dimensions: {heat_map_summary['quadrant_dimensions_mm']} mm")
print(f"MOOSE heat-map file: {heat_map_summary['output_path']}")


CSV total power: 1492.800 W
Lower-left quadrant power: 373.200 W
Quadrant map dimensions: (26.075, 27.05) mm
MOOSE heat-map file: /Users/jagardiner/projects/cutthroat/moose_inputs/q_map_LL.dat


In [91]:
# -----------------------------------------------------------------------------
# Test 2: generate and mesh a lower-left quadrant with 10 evenly spaced fins,
# then render its MOOSE input. The heat-map file and mesh share quadrant axes.
# -----------------------------------------------------------------------------
TEST2_FIN_COUNT = 10
TEST2_FIN_THICKNESS_MM = 0.500
TEST2_INLET_TEMPERATURE_K = 318.15
TEST2_PRESSURE_PIN_PA = 0.0
TEST2_CHANNEL_DEPTH_MM = 2.0
TEST2_BASEPLATE_THICKNESS_MM = 2.0
TEST2_BASEPLATE_CONDUCTIVITY_W_MK = 390.0
TEST2_FIN_CONDUCTIVITY_W_MK = 380.0
# Initial HTC guesses only. Calibrate against the teammate's 3D simulations.
TEST2_FLOOR_HTC_W_M2K = 5000.0
TEST2_FIN_HTC_W_M2K = 5000.0
TEST2_BOUNDARY_LAYER_FRACTION = 0.15
TEST2_FIN_EFFICIENCY = MooseInputWriter.calculate_plate_fin_efficiency(
    h_w_m2k=TEST2_FIN_HTC_W_M2K,
    k_w_mk=TEST2_FIN_CONDUCTIVITY_W_MK,
    thickness_mm=TEST2_FIN_THICKNESS_MM,
    height_mm=TEST2_CHANNEL_DEPTH_MM,
)

# Regenerate the LL heat map so this cell can run by itself after the class cells.
chip_power_map = ChipPowerMap(
    csv_path=HEAT_MAP_CSV_PATH,
    output_path=HEAT_MAP_OUTPUT_PATH,
    cell_size_mm=HEAT_MAP_CELL_SIZE_MM,
    chip_gap_mm=HEAT_MAP_CHIP_GAP_MM,
    full_domain_x_mm=FIN_PLATE_X_MM,
    full_domain_y_mm=FIN_PLATE_Y_MM,
    hbm_shape_cells=(5, 6),
    gpu_shape_cells=(15, 13),
    expected_full_power_w=HEAT_MAP_EXPECTED_POWER_W,
)
heat_map_summary = chip_power_map.export_moose()

# The full design dimensions are split at the two mirror planes.
TEST2_QUADRANT_X_MM = FIN_PLATE_X_MM / 2.0
TEST2_QUADRANT_Y_MM = FIN_PLATE_Y_MM / 2.0
test2_mesh_generator = MeshGenerator(
    element_size_um=ELEMENT_SIZE_UM,
    size_min_um=MESH_SIZE_MIN_UM,
    size_max_um=MESH_SIZE_MAX_UM,
    refine_dist_mm=MESH_REFINEMENT_DISTANCE_MM,
    refine_outer_walls=REFINE_OUTER_WALLS,
    verbose=MESH_VERBOSE,
)
test2_generator = Generator(
    fin_plate_x_mm=TEST2_QUADRANT_X_MM,
    fin_plate_y_mm=TEST2_QUADRANT_Y_MM,
    element_size_um=ELEMENT_SIZE_UM,
    min_feature_mm=MIN_FEATURE_MM,
    max_fin_thickness_mm=MAX_FIN_THICKNESS_MM,
    target_solid_fraction=TARGET_SOLID_FRACTION,
    solid_fraction_tolerance=SOLID_FRACTION_TOLERANCE,
    chip_gap_mm=CHIP_GAP_MM,
    seed=RANDOM_SEED,
    base_dir=BASE_DIR,
    mesh_generator=test2_mesh_generator,
    programmatic_layout="parallel_fins",
    programmatic_n_fins=TEST2_FIN_COUNT,
    programmatic_fin_thickness_mm=TEST2_FIN_THICKNESS_MM,
    programmatic_inlet_length_mm=PROGRAMMATIC_INLET_LENGTH_MM,
    programmatic_outlet_length_mm=PROGRAMMATIC_OUTLET_LENGTH_MM,
)
test2_design = test2_generator.generate(method="programmatic")
test2_mesh_file = test2_design.generate_mesh()

TEST2_BOUNDARY_CONDITIONS = {
    "U_in": U_INLET_M_S,
    "U_profile_type": "convergent",
    "boundary_layer_fraction": TEST2_BOUNDARY_LAYER_FRACTION,
    "domain_height": test2_design.actual_dimensions_mm[1] / 1000.0,
    "T_in": TEST2_INLET_TEMPERATURE_K,
    "P_pin": TEST2_PRESSURE_PIN_PA,
}
TEST2_MODEL_PARAMETERS = {
    "channel_depth_m": TEST2_CHANNEL_DEPTH_MM / 1000.0,
    "baseplate_thickness_m": TEST2_BASEPLATE_THICKNESS_MM / 1000.0,
    "baseplate_conductivity": TEST2_BASEPLATE_CONDUCTIVITY_W_MK,
    "h_floor_w_m2k": TEST2_FLOOR_HTC_W_M2K,
    "h_fin_w_m2k": TEST2_FIN_HTC_W_M2K,
    "fin_efficiency": TEST2_FIN_EFFICIENCY,
    "chip_power_map_file": heat_map_summary["output_path"],
}
test2_writer = MooseInputWriter(
    fluid_properties=FLUID_PROPERTIES,
    solid_properties={"k": TEST2_FIN_CONDUCTIVITY_W_MK},
    boundary_conditions=TEST2_BOUNDARY_CONDITIONS,
    test2_parameters=TEST2_MODEL_PARAMETERS,
)
test2_input_file = test2_writer.create_input_file(
    mesh_name=test2_mesh_file,
    output_dir=MOOSE_OUTPUT_DIR,
    boundary_conditions=TEST2_BOUNDARY_CONDITIONS,
    template="TEST2",
)
print(f"Test 2 fin count: {TEST2_FIN_COUNT}")
print(f"Fin thickness: {TEST2_FIN_THICKNESS_MM:.3f} mm")
print(f"Fin efficiency estimate: {TEST2_FIN_EFFICIENCY:.4f}")
print(f"MOOSE heat map: {heat_map_summary['output_path']}")
print(f"Resolved Test 2 mesh: {Path(test2_mesh_file).resolve()}")
print(f"Resolved Test 2 input file: {test2_input_file.resolve()}")


Generated programmatic design: solid density=11.4%
Test 2 fin count: 10
Fin thickness: 0.500 mm
Fin efficiency estimate: 0.9353
MOOSE heat map: /Users/jagardiner/projects/cutthroat/moose_inputs/q_map_LL.dat
Resolved Test 2 mesh: /Users/jagardiner/projects/cutthroat/meshes/3bf26755fd2a.msh
Resolved Test 2 input file: /Users/jagardiner/projects/cutthroat/moose_inputs/3bf26755fd2a.i
